# FIGNN Final  Validation

## 0. Environment
Sanity checks

In [1]:
import sys, importlib.util
print(sys.executable)
req=["numpy","pandas","scipy","sklearn","matplotlib","torch","torch_geometric"]
missing=[x for x in req if importlib.util.find_spec(x) is None]
print("Missing:",missing)
if missing: print("STOP: install/use the project fignn_env before continuing.")

d:\Portfolio Projects\QML EEG Paper\fignn_cni__project\fignn_env\Scripts\python.exe
Missing: []


## 1. Paths and authoritative files

In [4]:
from pathlib import Path
import os, json, copy, itertools, warnings
from scipy.io import loadmat

import numpy as np
import pandas as pd

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]

ROOT = next(
    (
        p for p in candidates
        if (p / "scripts").is_dir()
        and (p / "data").is_dir()
        and (p / "src").is_dir()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate the fignn_cni__project repository root."
    )

OUT = ROOT / "data" / "outputs"
PROC = ROOT / "data" / "data" / "processed"

S1 = OUT / "evaluation_loso_all29"
S2 = OUT / "evaluation_stage2_loso29"
LOSO = OUT / "loso"

TH = OUT / "plv_threshold_sensitivity"
EXP = OUT / "final_non_gpu_validation"
EXP.mkdir(parents=True, exist_ok=True)

GRAPH = PROC / "all_graphs_29sub.pkl"

SUBJECTS = list(range(1, 30))

S1CSV = S1 / "loso_all29_subject_metrics.csv"
S2CSV = S2 / "stage2_loso29_subject_metrics.csv"

RAW = ROOT / "data" / "data" / "raw"
NOTEBOOK_MAT = RAW / "notebook.mat"
TRIGGER_LIST = RAW / "trigger_list.txt"

METADATA_OUT = EXP / "dataset_metadata"
METADATA_OUT.mkdir(parents=True, exist_ok=True)

print("RAW           :", RAW)
print("NOTEBOOK MAT  :", NOTEBOOK_MAT)
print("TRIGGER LIST  :", TRIGGER_LIST)
print("METADATA OUT  :", METADATA_OUT)

print("ROOT :", ROOT)
print("GRAPH:", GRAPH)
print("EXP  :", EXP)

assert S1CSV.exists(), f"Missing: {S1CSV}"
assert S2CSV.exists(), f"Missing: {S2CSV}"
assert GRAPH.exists(), f"Missing: {GRAPH}"

print("All required files found.")

RAW           : D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw
NOTEBOOK MAT  : D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\notebook.mat
TRIGGER LIST  : D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\trigger_list.txt
METADATA OUT  : D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\outputs\final_non_gpu_validation\dataset_metadata
ROOT : D:\Portfolio Projects\QML EEG Paper\fignn_cni__project
GRAPH: D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\processed\all_graphs_29sub.pkl
EXP  : D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\outputs\final_non_gpu_validation
All required files found.


## 2. Load and audit current Stage-1/Stage-2 results

In [3]:
stage1=pd.read_csv(S1CSV).sort_values("subject").reset_index(drop=True)
stage2=pd.read_csv(S2CSV)

need=["subject","d01","d02","d12","d01_d02","path_efficiency","relative_1back_position","perpendicular_distance","oneback_between","procrustes_similarity","direction_cosine"]
assert not [c for c in need if c not in stage1.columns]
stage1["primary_pass"]=(stage1.relative_1back_position>=0)&(stage1.relative_1back_position<=1)
stage1["distance_order_pass"]=stage1.d01<stage1.d02

print("Primary",stage1.primary_pass.sum(),"/29")
print("d01<d02",stage1.distance_order_pass.sum(),"/29")
print("Stage2 columns:",stage2.columns.tolist())

display(stage1.head())
display(stage2.head())

Primary 23 /29
d01<d02 27 /29
Stage2 columns: ['subject', 'procrustes_similarity', 'spearman_rho', 'spearman_p', 'monotonic', 'd01', 'd02', 'd12', 'position_0back', 'position_1back', 'position_2back']


,subject,d01,d02,d12,d01_d02,d12_d02,angle_deg,path_efficiency,relative_1back_position,perpendicular_distance,centrality,oneback_between,procrustes_similarity,direction_cosine,n_train_trajectories,checkpoint,primary_pass,distance_order_pass
0,1,0.012123,0.021680,0.014933,0.559199,0.688780,73.957670,0.801296,0.419143,0.008025,0.838286,True,0.903420,-0.834839,28,/lp-dev/24BCE1793/outputs/loso/subject_01/fign...,True,True
1,2,0.024682,0.088463,0.077595,0.279016,0.877146,71.814190,0.864931,0.154232,0.020569,0.308465,True,0.962859,0.875531,28,/lp-dev/24BCE1793/outputs/loso/subject_02/fign...,True,True
2,3,0.017334,0.044216,0.033566,0.392028,0.759139,63.021250,0.868684,0.288697,0.011727,0.577395,True,0.965416,0.854275,28,/lp-dev/24BCE1793/outputs/loso/subject_03/fign...,True,True
3,4,0.043441,0.067354,0.028198,0.644963,0.418652,40.807743,0.940190,0.620354,0.011885,0.759292,True,0.809539,0.954822,28,/lp-dev/24BCE1793/outputs/loso/subject_04/fign...,True,True
4,5,0.032191,0.059927,0.033896,0.537167,0.565616,49.884975,0.906797,0.484313,0.013924,0.968627,True,0.912726,0.942994,28,/lp-dev/24BCE1793/outputs/loso/subject_05/fign...,True,True


,subject,procrustes_similarity,spearman_rho,spearman_p,monotonic,d01,d02,d12,position_0back,position_1back,position_2back
0,1,0.743220,0.177183,1.587841e-07,True,0.744418,0.012222,0.622532,0.796323,1.011168,1.154282
1,2,0.832842,0.383912,1.001761e-31,True,0.436197,0.075025,1.236057,1.116781,1.153905,1.689677
2,3,0.829298,0.163560,1.342913e-06,True,0.674193,0.026549,0.576390,1.066949,1.271790,1.411831
3,4,0.743939,0.319065,6.747242e-22,True,0.680980,0.028899,0.430563,0.764629,1.111964,1.390994
4,5,0.899522,0.151266,7.985097e-06,True,0.684313,0.016537,0.640976,0.746838,0.911126,1.083887


## 3. Exact six-labeling rho permutation null
This uses only the authoritative pairwise distances.

In [4]:
def t_from_d(dAB, dAC, dBC):
    return (
        (dAB * dAB + dAC * dAC - dBC * dBC)
        / (2 * dAC * dAC)
        if dAC > 1e-15
        else np.nan
    )


rows = []

for _, r in stage1.iterrows():

    D = {
        (0, 1): r.d01,
        (0, 2): r.d02,
        (1, 2): r.d12
    }

    def d(a, b):
        return D[tuple(sorted((a, b)))]

    for p in itertools.permutations([0, 1, 2]):
        A, B, C = p

        rho = t_from_d(
            d(A, B),
            d(A, C),
            d(B, C)
        )

        rows.append({
            "subject": int(r.subject),
            "permutation": "-".join(map(str, p)),
            "rho": rho,
            "pass": 0 <= rho <= 1
        })


perm = pd.DataFrame(rows)

null_counts = (
    perm.groupby("permutation")["pass"]
        .sum()
        .reset_index(name="n_pass")
)

print(
    "Observed:",
    int(stage1.primary_pass.sum()),
    "/29"
)

display(null_counts)

print(
    "Global six-permutation p:",
    np.mean(
        null_counts.n_pass >= int(stage1.primary_pass.sum())
    )
)

perm.to_csv(
    EXP / "rho_exact_permutations.csv",
    index=False
)

null_counts.to_csv(
    EXP / "rho_permutation_counts.csv",
    index=False
)

Observed: 23 /29


,permutation,n_pass
0,0-1-2,23
1,0-2-1,4
2,1-0-2,8
3,1-2-0,4
4,2-0-1,8
5,2-1-0,23


Global six-permutation p: 0.3333333333333333


## 4. Load graphs + exact final FIGNN model

In [5]:
import pickle
import torch

with open(GRAPH, "rb") as f:
    all_graphs = pickle.load(f)

print(type(all_graphs), len(all_graphs))

g0 = all_graphs[0]

print(g0)
print("x:", g0.x.shape)
print("edge:", g0.edge_index.shape)
print("edge_attr:", g0.edge_attr.shape)

def attr(g, names):
    for n in names:
        if hasattr(g, n):
            return getattr(g, n)
    return None


def subject(g):
    value = attr(g, ["subject", "subject_id", "subj"])
    return int(torch.as_tensor(value).view(-1)[0])


def label(g):
    value = attr(g, ["y", "label"])
    return int(torch.as_tensor(value).view(-1)[0])


ss = np.array([subject(g) for g in all_graphs])
yy = np.array([label(g) for g in all_graphs])

print("Subjects:", np.unique(ss))
print("Labels:", np.unique(yy))

print("\nGraphs per subject:")
print(pd.Series(ss).value_counts().sort_index())

print("\nGraphs per label:")
print(pd.Series(yy).value_counts().sort_index())

d:\Portfolio Projects\QML EEG Paper\fignn_cni__project\fignn_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<class 'list'> 27666
Data(x=[62, 4], edge_index=[2, 936], edge_attr=[936, 4], y=[1], subject=1, session=1)
x: torch.Size([62, 4])
edge: torch.Size([2, 936])
edge_attr: torch.Size([936, 4])
Subjects: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29]
Labels: [0 1 2 3]

Graphs per subject:
1     954
2     954
3     954
4     954
5     954
6     954
7     954
8     954
9     954
10    954
11    954
12    954
13    954
14    954
15    954
16    954
17    954
18    954
19    954
20    954
21    954
22    954
23    954
24    954
25    954
26    954
27    954
28    954
29    954
Name: count, dtype: int64

Graphs per label:
0    8352
1    8352
2    8352
3    2610
Name: count, dtype: int64


## 5. Frozen fold inference helpers

In [6]:
def norm_stats(s):
    p = LOSO / f"subject_{s:02d}" / "normalisation_stats.pt"
    assert p.exists(), p

    o = torch.load(
        p,
        map_location="cpu",
        weights_only=False
    )

    if isinstance(o, dict):
        mean = o.get("mean", o.get("x_mean", o.get("node_mean")))
        std = o.get("std", o.get("x_std", o.get("node_std")))
    else:
        mean, std = o[:2]

    return (
        torch.as_tensor(mean).float(),
        torch.as_tensor(std).float()
    )


def norm_graphs(gs, mean, std):
    out = []

    for g in gs:
        h = copy.deepcopy(g)
        h.x = (h.x.float() - mean) / std.clamp_min(1e-8)
        out.append(h)

    return out


def load_model(s):
    p = LOSO / f"subject_{s:02d}" / "fignn_best.pth"
    assert p.exists(), p

    m = FIGNNEncoder(
        node_in_dim=4,
        edge_dim=4,
        hidden_dim=128,
        latent_dim=64,
        n_heads=8,
        dropout=0.3
    )

    ck = torch.load(
        p,
        map_location="cpu",
        weights_only=False
    )

    m.load_state_dict(
        ck.get("model_state_dict", ck)
    )

    m.eval()
    return m


@torch.no_grad()
def encode(m, gs):
    zs = []
    ys = []
    ss = []

    for b in DataLoader(
        gs,
        batch_size=BATCH,
        shuffle=False,
        num_workers=0
    ):
        z = m(b)

        zs.append(z.cpu())
        ys.append(b.y.view(-1).cpu())
        ss.append(b.subject.view(-1).cpu())

    return (
        torch.cat(zs).numpy(),
        torch.cat(ys).numpy(),
        torch.cat(ss).numpy()
    )


print("helpers ready")

helpers ready


## 6. Cache 29 frozen-fold embeddings

In [7]:
EMB = EXP / "fold_embeddings"
EMB.mkdir(exist_ok=True)


def cache_path(s):
    return EMB / f"fold_{s:02d}.npz"


def run_fold(s):
    p = cache_path(s)

    if p.exists():
        a = np.load(p)
        return a["Z"], a["Y"], a["S"]

    # Outer LOSO split
    tr = [g for g in all_graphs if subject(g) != s]
    te = [g for g in all_graphs if subject(g) == s]

    # Fold-specific normalization
    mean, std = norm_stats(s)

    tr = norm_graphs(tr, mean, std)
    te = norm_graphs(te, mean, std)

    # Fold-specific frozen model
    m = load_model(s)

    # Encode training and held-out subjects
    z1, y1, s1 = encode(m, tr)
    z2, y2, s2 = encode(m, te)

    Z = np.r_[z1, z2]
    Y = np.r_[y1, y2]
    S = np.r_[s1, s2]

    np.savez_compressed(
        p,
        Z=Z,
        Y=Y,
        S=S
    )

    return Z, Y, S


for i, s in enumerate(SUBJECTS, 1):

    if cache_path(s).exists():
        print(f"[{i}/29] cached {s}")
    else:
        print(f"[{i}/29] encoding {s}")
        run_fold(s)


print(
    "caches:",
    len(list(EMB.glob("fold_*.npz")))
)

[1/29] cached 1
[2/29] cached 2
[3/29] cached 3
[4/29] cached 4
[5/29] cached 5
[6/29] cached 6
[7/29] cached 7
[8/29] cached 8
[9/29] cached 9
[10/29] cached 10
[11/29] cached 11
[12/29] cached 12
[13/29] cached 13
[14/29] cached 14
[15/29] cached 15
[16/29] cached 16
[17/29] cached 17
[18/29] cached 18
[19/29] cached 19
[20/29] cached 20
[21/29] cached 21
[22/29] cached 22
[23/29] cached 23
[24/29] cached 24
[25/29] cached 25
[26/29] cached 26
[27/29] cached 27
[28/29] cached 28
[29/29] cached 29
caches: 29


## 7. Verify regenerated geometry against the authoritative CSV

In [8]:
def cents(Z, Y):
    return np.stack([
        Z[Y == c].mean(0)
        for c in [0, 1, 2]
    ])


def rho(C):
    v = C[2] - C[0]

    return float(
        np.dot(C[1] - C[0], v)
        / np.dot(v, v)
    )


check = []

for s in SUBJECTS:

    a = np.load(cache_path(s))

    Z = a["Z"]
    Y = a["Y"]
    S = a["S"]

    C = cents(
        Z[S == s],
        Y[S == s]
    )

    rr = rho(C)

    csv_rho = stage1.loc[
        stage1.subject == s,
        "relative_1back_position"
    ].iloc[0]

    check.append([
        s,
        csv_rho,
        rr
    ])


check = pd.DataFrame(
    check,
    columns=[
        "subject",
        "csv_rho",
        "recomputed_rho"
    ]
)

check["abs_diff"] = abs(
    check["csv_rho"] - check["recomputed_rho"]
)

display(check)

print(
    "max rho diff:",
    check["abs_diff"].max()
)

assert check["abs_diff"].max() < 1e-4

print("Rho verification PASSED.")

,subject,csv_rho,recomputed_rho,abs_diff
0,1,0.419143,0.419142,8.635560e-07
1,2,0.154232,0.154232,1.670825e-07
2,3,0.288697,0.288697,3.550887e-07
3,4,0.620354,0.620354,2.951723e-07
4,5,0.484313,0.484313,4.186329e-07
5,6,0.469133,0.469133,1.823318e-07
6,7,0.299064,0.299064,1.802113e-07
7,8,0.427003,0.427004,3.608690e-07
8,9,0.002998,0.002997,2.286166e-07
9,10,0.120020,0.120020,2.905453e-07


max rho diff: 1.4589976501527824e-06
Rho verification PASSED.


## 8. Bootstrap rho CIs

In [9]:
rng = np.random.default_rng(20261002)
NBOOT = 1000


def boot_rho(Z, Y):
    A = [
        Z[Y == c]
        for c in [0, 1, 2]
    ]

    vals = np.empty(NBOOT)

    for i in range(NBOOT):
        C = [
            a[
                rng.integers(0, len(a), len(a))
            ].mean(0)
            for a in A
        ]

        vals[i] = rho(np.stack(C))

    return vals


br = []

for s in SUBJECTS:

    a = np.load(cache_path(s))

    z = a["Z"]
    y = a["Y"]
    ss = a["S"]

    v = boot_rho(
        z[ss == s],
        y[ss == s]
    )

    br.append([
        s,
        rho(cents(z[ss == s], y[ss == s])),
        np.percentile(v, 2.5),
        np.percentile(v, 97.5)
    ])


br = pd.DataFrame(
    br,
    columns=[
        "subject",
        "rho",
        "ci_low",
        "ci_high"
    ]
)

br["ci_inside_01"] = (
    (br["ci_low"] >= 0) &
    (br["ci_high"] <= 1)
)

display(br)

print(
    "CI entirely inside [0,1]:",
    br["ci_inside_01"].sum(),
    "/29"
)

br.to_csv(
    EXP / "rho_bootstrap_95CI.csv",
    index=False
)

,subject,rho,ci_low,ci_high,ci_inside_01
0,1,0.419142,-0.195996,1.124911,False
1,2,0.154232,0.033262,0.266840,True
2,3,0.288697,0.044359,0.544432,True
3,4,0.620354,0.453402,0.763586,True
4,5,0.484313,0.173194,0.774748,True
5,6,0.469133,0.237013,0.700579,True
6,7,0.299064,-0.043107,0.620547,False
7,8,0.427004,0.202859,0.641396,True
8,9,0.002997,-0.222959,0.179728,False
9,10,0.120020,-0.080302,0.311104,False


CI entirely inside [0,1]: 14 /29


## 9. Within-class dispersion + silhouette + frozen kNN

In [10]:
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier

disp = []
sil = []
knn = []

for s in SUBJECTS:

    a = np.load(cache_path(s))

    Z = a["Z"]
    Y = a["Y"]
    S = a["S"]

    tr = S != s
    te = S == s

    zt = Z[te]
    yt = Y[te]

    task = np.isin(yt, [0, 1, 2])

    zt_task = zt[task]
    yt_task = yt[task]

    sil_value = silhouette_score(
        zt_task,
        yt_task
    )

    

    # Within-class dispersion
    ds = {"subject": s}

    for c in [0, 1, 2]:

        zc = zt[yt == c]
        C = zc.mean(0)

        ds[f"dispersion_{c}"] = np.mean(
            np.linalg.norm(zc - C, axis=1)
        )

    disp.append(ds)

    sil.append([
        s,
        sil_value
    ])

    # Frozen 5-NN classifier
    clf = KNeighborsClassifier(
        n_neighbors=5,
        weights="distance"
    )

    tr_task = tr & np.isin(Y, [0, 1, 2])
    te_task = te & np.isin(Y, [0, 1, 2])

    clf.fit(Z[tr_task], Y[tr_task])

    acc = np.mean(
        clf.predict(Z[te_task]) == Y[te_task]
    )

    knn.append([
        s,
        np.mean(clf.predict(zt) == yt)
    ])


disp = pd.DataFrame(disp)

sil = pd.DataFrame(
    sil,
    columns=["subject", "silhouette"]
)

knn = pd.DataFrame(
    knn,
    columns=["subject", "knn_accuracy"]
)


display(disp)

print(
    "silhouette:",
    sil["silhouette"].mean(),
    sil["silhouette"].std(ddof=1),
    "positive:",
    (sil["silhouette"] > 0).sum()
)

print(
    "kNN:",
    knn["knn_accuracy"].mean(),
    knn["knn_accuracy"].std(ddof=1)
)


disp.to_csv(
    EXP / "within_class_dispersion.csv",
    index=False
)

sil.to_csv(
    EXP / "silhouette_loso.csv",
    index=False
)

knn.to_csv(
    EXP / "frozen_knn_loso.csv",
    index=False
)

,subject,dispersion_0,dispersion_1,dispersion_2
0,1,0.133437,0.141372,0.127464
1,2,0.094286,0.082217,0.069499
2,3,0.113978,0.107707,0.111115
3,4,0.104818,0.101526,0.092013
4,5,0.153193,0.148225,0.139006
5,6,0.100138,0.113088,0.104567
6,7,0.108619,0.112257,0.106341
7,8,0.105357,0.098638,0.085080
8,9,0.091491,0.087492,0.088729
9,10,0.113530,0.116300,0.117871


silhouette: -0.01457047969472177 0.011510064675205075 positive: 2
kNN: 0.3837923805392901 0.15411777347925182


## 10. Stage-2 centroids + translation + Ridge

In [11]:
from sklearn.linear_model import Ridge


def subj_cents(Z, Y, S, s):
    m = S == s

    return {
        name: Z[m & (Y == c)].mean(0)
        for c, name in [
            (0, "C0"),
            (1, "C1"),
            (2, "C2"),
            (3, "R")
        ]
        if np.any(m & (Y == c))
    }


rows = []

for s in SUBJECTS:

    a = np.load(cache_path(s))

    Z = a["Z"]
    Y = a["Y"]
    S = a["S"]

    cs = subj_cents(Z, Y, S, s)

    tr_subjects = [
        r for r in SUBJECTS
        if r != s
    ]

    X = []
    T = []

    for r in tr_subjects:

        cr = subj_cents(Z, Y, S, r)

        if all(
            k in cr
            for k in ["R", "C1", "C2"]
        ):
            X.append(
                np.r_[cr["R"], cr["C2"]]
            )
            T.append(cr["C1"])

    X = np.asarray(X)
    T = np.asarray(T)

    # Ridge baseline
    ridge = Ridge(alpha=1.0).fit(X, T)

    pred_r = ridge.predict(
        np.r_[cs["R"], cs["C2"]][None]
    )[0]

    # Population centroids
    pop1 = T.mean(0)

    pop0 = np.mean([
        subj_cents(Z, Y, S, r)["C0"]
        for r in tr_subjects
    ], axis=0)

    pop2 = np.mean([
        subj_cents(Z, Y, S, r)["C2"]
        for r in tr_subjects
    ], axis=0)

    # Translation baseline
    pred_trans = (
        pop1
        + (cs["C2"] - pop2)
    )

    # Midpoint baseline
    pred_mid = (
        cs["R"]
        + 0.5 * (cs["C2"] - cs["R"])
    )

    # Subject's actual 0→2 trajectory length
    d02 = np.linalg.norm(
        cs["C2"] - cs["C0"]
    )

    rows.append({
        "subject": s,
        "d02": d02,

        "population_error": np.linalg.norm(
            pop1 - cs["C1"]
        ),

        "midpoint_error": np.linalg.norm(
            pred_mid - cs["C1"]
        ),

        "translation_error": np.linalg.norm(
            pred_trans - cs["C1"]
        ),

        "ridge_error": np.linalg.norm(
            pred_r - cs["C1"]
        )
    })


st2 = pd.DataFrame(rows)


# Normalize all newly computed errors by subject-specific d02
for c in [
    "population_error",
    "midpoint_error",
    "translation_error",
    "ridge_error"
]:
    st2[c + "_normalized"] = (
        st2[c] / st2["d02"]
    )


# Load the authoritative LONG-format Stage-2 results
STAGE2_LONG = OUT / "stage2_calibration_comparison_64D.csv"
stage2_long = pd.read_csv(STAGE2_LONG)


# Add existing frozen-geometric error
fg = (
    stage2_long[
        stage2_long["method"] == "frozen_geometric"
    ][
        ["subject", "error_64D"]
    ]
    .rename(columns={
        "error_64D": "frozen_geometric_error"
    })
)


st2 = st2.merge(
    fg,
    on="subject",
    how="left"
)


# Normalize frozen-geometric error by d02
st2["frozen_geometric_normalized"] = (
    st2["frozen_geometric_error"]
    / st2["d02"]
)


display(st2)


st2.to_csv(
    EXP / "stage2_new_baselines_normalized.csv",
    index=False
)

,subject,d02,population_error,midpoint_error,translation_error,ridge_error,population_error_normalized,midpoint_error_normalized,translation_error_normalized,ridge_error_normalized,frozen_geometric_error,frozen_geometric_normalized
0,1,0.021680,0.054337,0.084958,0.082463,0.050368,2.506351,3.918773,3.803649,2.323267,0.056798,2.619850
1,2,0.088463,0.071156,0.026146,0.033822,0.071014,0.804367,0.295560,0.382329,0.802757,0.024916,0.281657
2,3,0.044216,0.139185,0.065468,0.041302,0.134789,3.147883,1.480650,0.934097,3.048456,0.036172,0.818088
3,4,0.067354,0.027844,0.086676,0.044073,0.027277,0.413401,1.286872,0.654351,0.404976,0.042476,0.630636
4,5,0.059927,0.051690,0.116989,0.056535,0.053905,0.862549,1.952176,0.943393,0.899505,0.077678,1.296205
5,6,0.059674,0.037841,0.036955,0.037671,0.037527,0.634132,0.619277,0.631273,0.628871,0.019430,0.325609
6,7,0.036039,0.093353,0.084817,0.045927,0.091693,2.590320,2.353459,1.274368,2.544254,0.045306,1.257117
7,8,0.045485,0.113199,0.050302,0.051728,0.110794,2.488682,1.105884,1.137250,2.435814,0.027177,0.597498
8,9,0.064458,0.056985,0.079453,0.015622,0.057810,0.884066,1.232634,0.242351,0.896859,0.029781,0.462019
9,10,0.069501,0.026162,0.015786,0.019033,0.026271,0.376421,0.227135,0.273855,0.377989,0.024018,0.345582


In [12]:
summary = pd.DataFrame({
    "method": [
        "Population",
        "Midpoint",
        "Translation",
        "Ridge",
        "Frozen geometric"
    ],
    "error_mean": [
        st2.population_error.mean(),
        st2.midpoint_error.mean(),
        st2.translation_error.mean(),
        st2.ridge_error.mean(),
        st2.frozen_geometric_error.mean()
    ],
    "error_sd": [
        st2.population_error.std(ddof=1),
        st2.midpoint_error.std(ddof=1),
        st2.translation_error.std(ddof=1),
        st2.ridge_error.std(ddof=1),
        st2.frozen_geometric_error.std(ddof=1)
    ],
    "normalized_mean": [
        st2.population_error_normalized.mean(),
        st2.midpoint_error_normalized.mean(),
        st2.translation_error_normalized.mean(),
        st2.ridge_error_normalized.mean(),
        st2.frozen_geometric_normalized.mean()
    ],
    "normalized_sd": [
        st2.population_error_normalized.std(ddof=1),
        st2.midpoint_error_normalized.std(ddof=1),
        st2.translation_error_normalized.std(ddof=1),
        st2.ridge_error_normalized.std(ddof=1),
        st2.frozen_geometric_normalized.std(ddof=1)
    ]
})

display(summary)

summary.to_csv(
    EXP / "stage2_baseline_summary.csv",
    index=False
)

,method,error_mean,error_sd,normalized_mean,normalized_sd
0,Population,0.064860,0.031883,1.591830,1.223564
1,Midpoint,0.052229,0.025477,1.201041,0.787674
2,Translation,0.041646,0.017665,1.028911,0.832316
3,Ridge,0.063370,0.030721,1.544754,1.163709
4,Frozen geometric,0.034542,0.016811,0.770288,0.523137


## 11. Band Weight Info

In [13]:
import torch
import numpy as np
import pandas as pd

bands = ["delta", "theta", "alpha", "beta"]

weight_rows = []

for s in SUBJECTS:

    p = LOSO / f"subject_{s:02d}" / "fignn_best.pth"

    ckpt = torch.load(
        p,
        map_location="cpu",
        weights_only=False
    )

    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        state = ckpt["model_state_dict"]
    else:
        state = ckpt

    # Find the BandWeightLayer parameter robustly
    candidates = [
        k for k in state.keys()
        if k.endswith("raw_weights")
    ]

    if len(candidates) != 1:
        print(f"Subject {s}: found keys =", candidates)
        raise RuntimeError(
            f"Could not uniquely identify raw_weights for subject {s}"
        )

    raw = state[candidates[0]].float()

    weights = torch.softmax(raw, dim=0).numpy()

    weight_rows.append({
        "subject": s,
        "delta": weights[0],
        "theta": weights[1],
        "alpha": weights[2],
        "beta": weights[3]
    })


band_weights = pd.DataFrame(weight_rows)

display(band_weights)

,subject,delta,theta,alpha,beta
0,1,0.236590,0.184988,0.218941,0.359481
1,2,0.229834,0.203581,0.243147,0.323438
2,3,0.202076,0.191467,0.236684,0.369772
3,4,0.213133,0.200441,0.233958,0.352468
4,5,0.229596,0.175564,0.248113,0.346727
5,6,0.225826,0.168821,0.244044,0.361309
6,7,0.203833,0.193011,0.300459,0.302697
7,8,0.215211,0.187392,0.238202,0.359195
8,9,0.216267,0.166751,0.260154,0.356828
9,10,0.232526,0.180467,0.242025,0.344982


In [14]:
band_summary = pd.DataFrame({
    "band": bands,
    "mean": [
        band_weights[b].mean()
        for b in bands
    ],
    "sd": [
        band_weights[b].std(ddof=1)
        for b in bands
    ],
    "min": [
        band_weights[b].min()
        for b in bands
    ],
    "max": [
        band_weights[b].max()
        for b in bands
    ]
})

display(band_summary)

band_weights.to_csv(
    EXP / "loso_band_weights_all29.csv",
    index=False
)

band_summary.to_csv(
    EXP / "loso_band_weights_summary.csv",
    index=False
)

print("\n29-fold band weights:")
for _, r in band_summary.iterrows():
    print(
        f"{r['band']:>5}: "
        f"{r['mean']:.4f} ± {r['sd']:.4f} "
        f"[{r['min']:.4f}, {r['max']:.4f}]"
    )

,band,mean,sd,min,max
0,delta,0.213044,0.014915,0.172031,0.236590
1,theta,0.195799,0.018439,0.166751,0.244059
2,alpha,0.248702,0.026063,0.206048,0.339445
3,beta,0.342455,0.026780,0.260995,0.387886



29-fold band weights:
delta: 0.2130 ± 0.0149 [0.1720, 0.2366]
theta: 0.1958 ± 0.0184 [0.1668, 0.2441]
alpha: 0.2487 ± 0.0261 [0.2060, 0.3394]
 beta: 0.3425 ± 0.0268 [0.2610, 0.3879]


## 12. Stage-2 improvement vs direction cosine

In [15]:
from scipy.stats import pearsonr, spearmanr

# =============================================================
# Stage 2 improvement vs Stage 1 trajectory direction
# =============================================================

# Stage-2 long-format results
stage2_long = pd.read_csv(
    OUT / "stage2_calibration_comparison_64D.csv"
)

population = (
    stage2_long[
        stage2_long["method"] == "population_only"
    ][["subject", "error_64D"]]
    .rename(columns={"error_64D": "population_error"})
)

frozen = (
    stage2_long[
        stage2_long["method"] == "frozen_geometric"
    ][["subject", "error_64D"]]
    .rename(columns={"error_64D": "frozen_error"})
)

corr_df = (
    population
    .merge(frozen, on="subject")
    .merge(
        stage1[["subject", "direction_cosine"]],
        on="subject"
    )
)

# Positive = frozen geometric improved over population-only
corr_df["improvement"] = (
    corr_df["population_error"]
    - corr_df["frozen_error"]
)

corr_df["improved"] = corr_df["improvement"] > 0


# =============================================================
# Correlation across all 29 subjects
# =============================================================

pearson_all = pearsonr(
    corr_df["direction_cosine"],
    corr_df["improvement"]
)

spearman_all = spearmanr(
    corr_df["direction_cosine"],
    corr_df["improvement"]
)


# =============================================================
# Correlation among the 7 non-improved subjects
# =============================================================

non_improved = corr_df[
    ~corr_df["improved"]
].copy()

pearson_non = pearsonr(
    non_improved["direction_cosine"],
    non_improved["improvement"]
)

spearman_non = spearmanr(
    non_improved["direction_cosine"],
    non_improved["improvement"]
)


# =============================================================
# Print results
# =============================================================

print("=" * 70)
print("STAGE 2 IMPROVEMENT vs STAGE 1 DIRECTION COSINE")
print("=" * 70)

print(
    f"Improved: "
    f"{corr_df['improved'].sum()}/29"
)

print(
    f"Non-improved: "
    f"{(~corr_df['improved']).sum()}/29"
)

print("\nAll 29 subjects:")
print(
    f"Pearson  r = {pearson_all.statistic:.4f}, "
    f"p = {pearson_all.pvalue:.4g}"
)
print(
    f"Spearman rho = {spearman_all.statistic:.4f}, "
    f"p = {spearman_all.pvalue:.4g}"
)

print("\nSeven non-improved subjects:")
print(
    f"Pearson  r = {pearson_non.statistic:.4f}, "
    f"p = {pearson_non.pvalue:.4g}"
)
print(
    f"Spearman rho = {spearman_non.statistic:.4f}, "
    f"p = {spearman_non.pvalue:.4g}"
)

print("\nNon-improved subjects:")
display(
    non_improved[
        [
            "subject",
            "direction_cosine",
            "population_error",
            "frozen_error",
            "improvement"
        ]
    ].sort_values("subject")
)


# =============================================================
# Save
# =============================================================

corr_df.to_csv(
    EXP / "stage2_improvement_vs_direction_cosine.csv",
    index=False
)


# =============================================================
# FINAL AUDIT
# =============================================================

audit = {
    "n_subjects": len(stage1),
    "primary_pass": int(stage1.primary_pass.sum()),
    "distance_order_pass": int(stage1.distance_order_pass.sum()),
    "mean_d01_d02": float(stage1.d01_d02.mean()),
    "mean_path_efficiency": float(stage1.path_efficiency.mean()),
    "mean_procrustes": float(
        stage1.procrustes_similarity.mean()
    ),
    "mean_perpendicular_distance": float(
        stage1.perpendicular_distance.mean()
    ),
    "stage2_improved": int(
        corr_df["improved"].sum()
    ),
    "stage2_non_improved": int(
        (~corr_df["improved"]).sum()
    ),
    "pearson_all_r": float(
        pearson_all.statistic
    ),
    "pearson_all_p": float(
        pearson_all.pvalue
    ),
    "spearman_all_rho": float(
        spearman_all.statistic
    ),
    "spearman_all_p": float(
        spearman_all.pvalue
    ),
    "pearson_non_improved_r": float(
        pearson_non.statistic
    ),
    "pearson_non_improved_p": float(
        pearson_non.pvalue
    ),
    "spearman_non_improved_rho": float(
        spearman_non.statistic
    ),
    "spearman_non_improved_p": float(
        spearman_non.pvalue
    )
}

print("\n" + "=" * 70)
print("FINAL AUDIT")
print("=" * 70)

print(json.dumps(audit, indent=2))

assert audit["n_subjects"] == 29
assert audit["primary_pass"] == 23
assert audit["distance_order_pass"] == 27
assert audit["stage2_improved"] == 22
assert audit["stage2_non_improved"] == 7

(EXP / "final_audit.json").write_text(
    json.dumps(audit, indent=2)
)

print("\nAll final exports written to:")
print(EXP)

STAGE 2 IMPROVEMENT vs STAGE 1 DIRECTION COSINE
Improved: 22/29
Non-improved: 7/29

All 29 subjects:
Pearson  r = -0.0060, p = 0.9753
Spearman rho = -0.3498, p = 0.0629

Seven non-improved subjects:
Pearson  r = -0.5664, p = 0.1849
Spearman rho = -0.5000, p = 0.2532

Non-improved subjects:


,subject,direction_cosine,population_error,frozen_error,improvement
0,1,-0.834839,0.054337,0.056798,-0.002461
3,4,0.954822,0.027844,0.042476,-0.014632
4,5,0.942994,0.051690,0.077678,-0.025988
18,19,0.888955,0.053983,0.065714,-0.011730
21,22,0.974870,0.050470,0.066246,-0.015775
22,23,0.602872,0.035595,0.046839,-0.011244
26,27,0.962827,0.032357,0.036350,-0.003993



FINAL AUDIT
{
  "n_subjects": 29,
  "primary_pass": 23,
  "distance_order_pass": 27,
  "mean_d01_d02": 0.47715064068965524,
  "mean_path_efficiency": 0.8191989051724137,
  "mean_procrustes": 0.87746762919339,
  "mean_perpendicular_distance": 0.013467512582758618,
  "stage2_improved": 22,
  "stage2_non_improved": 7,
  "pearson_all_r": -0.006013899437075029,
  "pearson_all_p": 0.9753002626705634,
  "spearman_all_rho": -0.3497536945812807,
  "spearman_all_p": 0.0629047444505732,
  "pearson_non_improved_r": -0.5664253504619371,
  "pearson_non_improved_p": 0.1849261845307234,
  "spearman_non_improved_rho": -0.5,
  "spearman_non_improved_p": 0.2531699951003227
}

All final exports written to:
D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\outputs\final_non_gpu_validation


## 13. Threshold sensitivity verification

In [16]:
for p in [TH/"threshold_summary.csv",TH/"threshold_results.csv",TH/"threshold_0.5_v4_check.csv"]:
    print("\\n",p.name,p.exists())
    if p.exists(): 
        display(pd.read_csv(p).head(20))

\n threshold_summary.csv True


,threshold,subjects,between_count,between_rate,mean_d01_d02,std_d01_d02,mean_d12_d02,std_d12_d02,mean_relative_position,std_relative_position,...,std_perpendicular_distance,mean_path_efficiency,std_path_efficiency,mean_angle,std_angle,mean_procrustes,std_procrustes,mean_direction_cosine,std_direction_cosine,between_rate_%
0,0.3,29,21,0.724138,0.628058,0.535069,0.900184,0.480973,0.318598,0.465085,...,0.012085,0.767363,0.217247,83.785517,38.088045,0.806646,0.218721,0.699435,0.446682,72.413793
1,0.4,29,23,0.793103,0.536103,0.392645,0.865381,0.386489,0.271577,0.343124,...,0.010628,0.803625,0.205214,75.428975,39.601466,0.847213,0.193901,0.766563,0.395966,79.310345
2,0.5,29,23,0.793103,0.477151,0.269485,0.823923,0.283942,0.270550,0.290557,...,0.008846,0.819199,0.170369,73.196998,39.671425,0.877468,0.154761,0.785395,0.403633,79.310345
3,0.6,29,23,0.793103,0.478270,0.247467,0.862629,0.346123,0.214036,0.361057,...,0.005877,0.800368,0.170305,78.062188,42.257198,0.878463,0.142283,0.767682,0.382217,79.310345
4,0.7,29,25,0.862069,0.548655,0.415302,0.880549,0.549644,0.200246,0.555329,...,0.004728,0.788329,0.178190,79.764054,40.089830,0.849583,0.150661,0.745407,0.367577,86.206897


\n threshold_results.csv True


,threshold,subject,d01,d02,d12,d01_d02,d12_d02,angle_deg,path_efficiency,d01_lt_d02,relative_1back_position,centrality,perpendicular_distance,oneback_between_x,procrustes_similarity,direction_cosine,checkpoint,n_training_subjects
0,0.3,1,0.010148,0.012563,0.013802,0.807773,1.098627,118.963020,0.524549,True,0.222758,0.445517,0.009755,True,0.721956,-0.141625,/lp-dev/24BCE1793/outputs/loso/subject_01/fign...,28
1,0.3,2,0.036216,0.182782,0.155488,0.198140,0.850678,45.307831,0.953454,True,0.157803,0.315606,0.021902,True,0.977746,0.898332,/lp-dev/24BCE1793/outputs/loso/subject_02/fign...,28
2,0.3,3,0.021589,0.046331,0.052413,0.465969,1.131293,118.122559,0.626071,True,-0.031349,0.000000,0.021540,False,0.831725,0.837136,/lp-dev/24BCE1793/outputs/loso/subject_03/fign...,28
3,0.3,4,0.043217,0.066045,0.028109,0.654354,0.425607,45.460278,0.925959,True,0.623519,0.752962,0.013110,True,0.801861,0.917611,/lp-dev/24BCE1793/outputs/loso/subject_04/fign...,28
4,0.3,5,0.057522,0.066552,0.019302,0.864309,0.290034,70.318802,0.866294,True,0.831455,0.337090,0.015709,True,0.615049,0.949180,/lp-dev/24BCE1793/outputs/loso/subject_05/fign...,28
5,0.3,6,0.045192,0.061421,0.025766,0.735764,0.419502,62.747837,0.865602,True,0.682683,0.634633,0.016854,True,0.745569,0.959697,/lp-dev/24BCE1793/outputs/loso/subject_06/fign...,28
6,0.3,7,0.006750,0.027793,0.024110,0.242859,0.867479,63.425045,0.900626,True,0.153230,0.306460,0.005237,True,0.974414,0.721066,/lp-dev/24BCE1793/outputs/loso/subject_07/fign...,28
7,0.3,8,0.031642,0.075783,0.047140,0.417530,0.622036,32.366241,0.961940,True,0.393701,0.787403,0.010537,True,0.972671,0.901933,/lp-dev/24BCE1793/outputs/loso/subject_08/fign...,28
8,0.3,9,0.011728,0.087286,0.088729,0.134368,1.016538,100.845879,0.868881,True,-0.007647,0.000000,0.011709,False,0.962395,0.976421,/lp-dev/24BCE1793/outputs/loso/subject_09/fign...,28
9,0.3,10,0.019361,0.082081,0.068589,0.235872,0.835632,51.372608,0.933267,True,0.178677,0.357354,0.012639,True,0.984649,0.955218,/lp-dev/24BCE1793/outputs/loso/subject_10/fign...,28


\n threshold_0.5_v4_check.csv True


,threshold,subject,d01,d02,d12,d01_d02,d12_d02,angle_deg,path_efficiency,d01_lt_d02,relative_1back_position,centrality,perpendicular_distance,oneback_between_x,procrustes_similarity,direction_cosine,checkpoint,n_training_subjects
0,0.5,1,0.012123,0.021680,0.014933,0.559199,0.688781,106.042252,0.801295,True,0.419142,0.838285,0.008025,True,0.903420,-0.834838,/lp-dev/24BCE1793/outputs/loso/subject_01/fign...,28
1,0.5,2,0.024683,0.088463,0.077595,0.279016,0.877146,108.185791,0.864931,True,0.154232,0.308465,0.020569,True,0.962859,0.875531,/lp-dev/24BCE1793/outputs/loso/subject_02/fign...,28
2,0.5,3,0.017334,0.044216,0.033566,0.392029,0.759139,116.978569,0.868683,True,0.288697,0.577394,0.011727,True,0.965416,0.854275,/lp-dev/24BCE1793/outputs/loso/subject_03/fign...,28
3,0.5,4,0.043441,0.067354,0.028198,0.644963,0.418652,139.192261,0.940189,True,0.620354,0.759292,0.011885,True,0.809539,0.954823,/lp-dev/24BCE1793/outputs/loso/subject_04/fign...,28
4,0.5,5,0.032191,0.059927,0.033896,0.537167,0.565617,130.114899,0.906796,True,0.484313,0.968626,0.013924,True,0.912726,0.942994,/lp-dev/24BCE1793/outputs/loso/subject_05/fign...,28
5,0.5,6,0.032459,0.059674,0.035685,0.543945,0.598005,122.183678,0.875695,True,0.469133,0.938266,0.016428,True,0.907934,0.952020,/lp-dev/24BCE1793/outputs/loso/subject_06/fign...,28
6,0.5,7,0.011828,0.036039,0.025727,0.328198,0.713852,144.759933,0.959646,True,0.299065,0.598129,0.004872,True,0.998408,0.862118,/lp-dev/24BCE1793/outputs/loso/subject_07/fign...,28
7,0.5,8,0.020241,0.045485,0.026678,0.444999,0.586529,151.318634,0.969436,True,0.427004,0.854008,0.005698,True,0.964522,0.765699,/lp-dev/24BCE1793/outputs/loso/subject_08/fign...,28
8,0.5,9,0.010815,0.064458,0.065168,0.167778,1.011016,81.472893,0.848324,True,0.002998,0.005995,0.010813,True,0.943956,0.964994,/lp-dev/24BCE1793/outputs/loso/subject_09/fign...,28
9,0.5,10,0.012909,0.069501,0.061948,0.185734,0.891323,121.104790,0.928456,True,0.120020,0.240040,0.009852,True,0.974131,0.945760,/lp-dev/24BCE1793/outputs/loso/subject_10/fign...,28


## 14. Edge density at threshold 0.5

In [17]:
possible = 62 * 61 // 2
counts = []

for g in all_graphs:

    ei = g.edge_index.numpy()

    pairs = {
        tuple(sorted((int(a), int(b))))
        for a, b in zip(ei[0], ei[1])
        if a != b
    }

    counts.append(len(pairs))


counts = np.asarray(counts)

density = counts / possible

print("possible:", possible)
print("mean edges:", counts.mean())
print("sd edges:", counts.std(ddof=1))
print("density:", density.mean())
print("sd density:", density.std(ddof=1))


pd.DataFrame({
    "n_edges": counts,
    "density": density
}).to_csv(
    EXP / "edge_density_threshold_05.csv",
    index=False
)

possible: 1891
mean edges: 558.1646425215065
sd edges: 145.9443973175378
density: 0.2951690335914895
sd density: 0.07717842269568367


## 15. REST statistical test

In [18]:
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

rest_rows = []

for s in SUBJECTS:

    a = np.load(cache_path(s))

    Z = a["Z"]
    Y = a["Y"]
    S = a["S"]

    # Held-out subject only
    mask = S == s

    z = Z[mask]
    y = Y[mask]

    # Subject centroids
    R = z[y == 3].mean(axis=0)
    C0 = z[y == 0].mean(axis=0)
    C1 = z[y == 1].mean(axis=0)
    C2 = z[y == 2].mean(axis=0)

    rest_rows.append({
        "subject": s,
        "REST_0back": np.linalg.norm(C0 - R),
        "REST_1back": np.linalg.norm(C1 - R),
        "REST_2back": np.linalg.norm(C2 - R)
    })

rest_dist = pd.DataFrame(rest_rows)

# -------------------------------------------------------------
# Friedman omnibus test
# -------------------------------------------------------------

friedman_stat, friedman_p = friedmanchisquare(
    rest_dist["REST_0back"],
    rest_dist["REST_1back"],
    rest_dist["REST_2back"]
)

print("=" * 70)
print("REST → N-BACK DISTANCE ANALYSIS")
print("=" * 70)

print("\nMean ± SD:")
for c in [
    "REST_0back",
    "REST_1back",
    "REST_2back"
]:
    print(
        f"{c}: "
        f"{rest_dist[c].mean():.4f} ± "
        f"{rest_dist[c].std(ddof=1):.4f}"
    )

print(
    f"\nFriedman χ² = {friedman_stat:.4f}, "
    f"p = {friedman_p:.4g}"
)


# -------------------------------------------------------------
# Pairwise Wilcoxon tests
# -------------------------------------------------------------

pairs = [
    ("REST_0back", "REST_1back"),
    ("REST_0back", "REST_2back"),
    ("REST_1back", "REST_2back")
]

pair_results = []

for a_col, b_col in pairs:

    stat, p = wilcoxon(
        rest_dist[a_col],
        rest_dist[b_col],
        alternative="less"
    )

    pair_results.append({
        "comparison": f"{a_col} < {b_col}",
        "W": stat,
        "raw_p": p
    })

pair_results = pd.DataFrame(pair_results)

pair_results["holm_p"] = multipletests(
    pair_results["raw_p"],
    method="holm"
)[1]

print("\nPairwise Wilcoxon tests:")
display(pair_results)


# -------------------------------------------------------------
# Save
# -------------------------------------------------------------

rest_dist.to_csv(
    EXP / "rest_nback_distances.csv",
    index=False
)

pair_results.to_csv(
    EXP / "rest_nback_statistics.csv",
    index=False
)

REST → N-BACK DISTANCE ANALYSIS

Mean ± SD:
REST_0back: 0.1136 ± 0.0493
REST_1back: 0.1245 ± 0.0537
REST_2back: 0.1561 ± 0.0565

Friedman χ² = 37.4483, p = 7.383e-09

Pairwise Wilcoxon tests:


,comparison,W,raw_p,holm_p
0,REST_0back < REST_1back,71.0,4.873462e-04,4.873462e-04
1,REST_0back < REST_2back,5.0,1.862645e-08,5.587935e-08
2,REST_1back < REST_2back,7.0,3.539026e-08,7.078052e-08


## 16. COG-BCI Session Metadata Audit

This section audits the dataset-level session metadata stored in
`notebook.mat`, including session interruption flags, experimental block
order, and investigator notes.

The metadata audit is descriptive only. It is not used to modify the
EEG graphs, labels, embeddings, or statistical analyses.

In [19]:
# =============================================================
# COG-BCI session metadata audit
# =============================================================

from scipy.io import loadmat

assert NOTEBOOK_MAT.exists(), f"Missing: {NOTEBOOK_MAT}"

nb_mat = loadmat(
    NOTEBOOK_MAT,
    squeeze_me=True,
    struct_as_record=False
)

assert "notebook" in nb_mat, "Expected variable 'notebook' not found."

notebook = nb_mat["notebook"]

print("Notebook fields:")
print(notebook._fieldnames)

Notebook fields:
['SBJ_1', 'SBJ_2', 'SBJ_3', 'SBJ_4', 'SBJ_5', 'SBJ_6', 'SBJ_7', 'SBJ_8', 'SBJ_9', 'SBJ_10', 'SBJ_11', 'SBJ_12', 'SBJ_13', 'SBJ_14', 'SBJ_15', 'SBJ_16', 'SBJ_17', 'SBJ_18', 'SBJ_19', 'SBJ_20', 'SBJ_21', 'SBJ_22', 'SBJ_23', 'SBJ_24', 'SBJ_25', 'SBJ_26', 'SBJ_27', 'SBJ_28', 'SBJ_29']


In [20]:
# =============================================================
# Extract subject/session metadata
# =============================================================

metadata_rows = []

for s in range(1, 30):

    subject_struct = getattr(
        notebook,
        f"SBJ_{s}"
    )

    for sess in range(1, 4):

        session_struct = getattr(
            subject_struct,
            f"SESS_{sess}"
        )

        interrupted = int(
            getattr(
                session_struct,
                "interrupted",
                0
            )
        )

        order = getattr(
            session_struct,
            "Order",
            None
        )

        if order is not None:
            order = np.asarray(order).flatten().astype(int).tolist()
            order_string = "-".join(map(str, order))
        else:
            order_string = ""

        notes = getattr(
            session_struct,
            "Notes",
            ""
        )

        if isinstance(notes, np.ndarray):
            notes = str(notes.tolist())

        notes = str(notes).strip()

        metadata_rows.append({
            "subject": s,
            "session": sess,
            "interrupted": interrupted,
            "order": order_string,
            "notes": notes
        })


session_metadata = pd.DataFrame(
    metadata_rows
)

display(session_metadata)

print(
    "Total subject-session records:",
    len(session_metadata)
)

print(
    "Interrupted sessions:",
    session_metadata["interrupted"].sum()
)

print(
    "Uninterrupted sessions:",
    (session_metadata["interrupted"] == 0).sum()
)

,subject,session,interrupted,order,notes
0,1,1,0,4-6-3-1-5-7-2-8,
1,1,2,0,8-6-3-1-7-2-5-4,
2,1,3,0,3-5-2-6-8-1-4-7,
3,2,1,1,5-7-3-6-4-2-8-1,the Two Back (position 5) had to be recorded l...
4,2,2,0,6-4-3-8-5-1-7-2,
...,...,...,...,...,...
82,28,2,0,5-1-2-3-8-4-6-7,
83,28,3,0,6-5-3-4-8-7-1-2,
84,29,1,0,4-6-2-1-8-5-7-3,
85,29,2,0,1-2-5-3-7-6-8-4,


Total subject-session records: 87
Interrupted sessions: 8
Uninterrupted sessions: 79


In [21]:
session_metadata.to_csv(
    METADATA_OUT / "cog_bci_session_metadata.csv",
    index=False
)

print(
    "Saved:",
    METADATA_OUT / "cog_bci_session_metadata.csv"
)

Saved: D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\outputs\final_non_gpu_validation\dataset_metadata\cog_bci_session_metadata.csv


In [22]:
# =============================================================
# Cross-reference Stage-1 geometric failures
# =============================================================

FAILED_SUBJECTS = (
    stage1.loc[
        ~stage1["primary_pass"],
        "subject"
    ]
    .astype(int)
    .tolist()
)

print("Stage-1 geometric failures:")
print(FAILED_SUBJECTS)

failure_metadata = session_metadata[
    session_metadata["subject"].isin(FAILED_SUBJECTS)
].copy()

display(
    failure_metadata[
        [
            "subject",
            "session",
            "interrupted",
            "order",
            "notes"
        ]
    ]
)

Stage-1 geometric failures:
[12, 14, 19, 22, 25, 29]


,subject,session,interrupted,order,notes
33,12,1,0,6-2-7-4-1-5-3-8,
34,12,2,1,1-3-6-2-7-4-5-8,Mistook what task the participant was doing ad...
35,12,3,0,1-8-6-5-4-3-2-7,
39,14,1,0,2-5-6-8-7-1-3-4,
40,14,2,0,2-4-7-6-3-8-1-5,
41,14,3,0,6-3-7-8-5-1-2-4,
54,19,1,0,5-2-3-6-4-1-7-8,
55,19,2,0,1-4-2-5-3-8-7-6,
56,19,3,0,4-1-7-8-5-2-3-6,
63,22,1,0,3-2-1-5-4-6-8-7,


In [23]:
failure_metadata.to_csv(
    METADATA_OUT / "stage1_failure_session_metadata.csv",
    index=False
)

In [24]:
# =============================================================
# Subject-level failure metadata summary
# =============================================================

failure_summary = []

for s in FAILED_SUBJECTS:

    sub = session_metadata[
        session_metadata["subject"] == s
    ]

    interrupted_sessions = sub.loc[
        sub["interrupted"] == 1,
        "session"
    ].astype(int).tolist()

    notes = sub.loc[
        sub["notes"].str.len() > 0,
        ["session", "notes"]
    ]

    failure_summary.append({
        "subject": s,
        "interrupted_sessions": ",".join(
            map(str, interrupted_sessions)
        ),
        "n_interrupted": len(interrupted_sessions),
        "documented_notes": " | ".join(
            f"S{int(r.session)}: {r.notes}"
            for _, r in notes.iterrows()
        )
    })

failure_summary = pd.DataFrame(
    failure_summary
)

display(failure_summary)

failure_summary.to_csv(
    METADATA_OUT / "stage1_failure_subject_summary.csv",
    index=False
)

,subject,interrupted_sessions,n_interrupted,documented_notes
0,12,2,1,S2: Mistook what task the participant was doin...
1,14,,0,
2,19,,0,
3,22,3,1,S3: []
4,25,,0,
5,29,3,1,S3: []


In [25]:
# =============================================================
# All documented session notes
# =============================================================

documented_notes = session_metadata[
    session_metadata["notes"].str.len() > 0
].copy()

print(
    "Sessions with documented notes:",
    len(documented_notes)
)

display(documented_notes)

documented_notes.to_csv(
    METADATA_OUT / "all_documented_session_notes.csv",
    index=False
)

Sessions with documented notes: 8


,subject,session,interrupted,order,notes
3,2,1,1,5-7-3-6-4-2-8-1,the Two Back (position 5) had to be recorded l...
13,5,2,1,2-1-6-8-7-4-5-3,Two pack had to be redone (11:34 new recodring...
21,8,1,1,4-1-8-3-5-6-7-2,Crash after Oneback (position 1) RSME was reco...
24,9,1,1,3-6-8-4-7-5-2-1,redid the two back conditiion after having mis...
34,12,2,1,1-3-6-2-7-4-5-8,Mistook what task the participant was doing ad...
46,16,2,1,5-6-1-2-7-3-4-8,[]
65,22,3,1,2-5-3-4-7-1-8-6,[]
86,29,3,1,7-4-8-1-6-5-3-2,[]


## 17. COG-BCI Trigger Semantics Audit

This section records the documented event-code semantics used to define
task epochs. The trigger definitions are treated as acquisition metadata
and are not inferred from the processed graph labels.

In [26]:
# =============================================================
# COG-BCI trigger semantics audit
# =============================================================

trigger_df = pd.read_csv(
    TRIGGER_LIST
)

trigger_df.columns = [
    "code",
    "description"
]

trigger_df["code"] = (
    trigger_df["code"]
    .astype(str)
    .str.strip()
)

trigger_df["description"] = (
    trigger_df["description"]
    .astype(str)
    .str.strip()
)

# Authoritative N-back trigger mapping used by preprocessing
EXPECTED_TRIGGERS = {
    "6021": "ZEROBACK Normal Trial Onset",
    "6121": "ONEBACK Normal Trial Onset",
    "6221": "TWOBACK Normal Trial Onset",

    "6022": "ZEROBACK Hit Trial Onset",
    "6122": "ONEBACK Hit Trial Onset",
    "6222": "TWOBACK Hit Trial Onset",

    "6023": "ZEROBACK Conflict Trial Onset",
    "6123": "ONEBACK Conflict Trial Onset",
    "6223": "TWOBACK Conflict Trial Onset",

    "6132": "ONEBACK Correct Response",
    "6232": "TWOBACK Correct Response",
}

for code, expected_description in EXPECTED_TRIGGERS.items():

    match = trigger_df[
        trigger_df["code"] == code
    ]

    assert len(match) == 1, (
        f"Expected exactly one entry for trigger {code}"
    )

    actual = match.iloc[0]["description"]

    assert actual == expected_description, (
        f"Trigger {code}: expected "
        f"'{expected_description}', got '{actual}'"
    )

trigger_audit = trigger_df[
    trigger_df["code"].isin(EXPECTED_TRIGGERS.keys())
].copy()

display(trigger_audit)

trigger_audit.to_csv(
    METADATA_OUT / "cog_bci_nback_trigger_audit.csv",
    index=False
)

print("Trigger semantics audit PASSED.")

,code,description
3,6021,ZEROBACK Normal Trial Onset
4,6022,ZEROBACK Hit Trial Onset
5,6023,ZEROBACK Conflict Trial Onset
13,6121,ONEBACK Normal Trial Onset
14,6122,ONEBACK Hit Trial Onset
15,6123,ONEBACK Conflict Trial Onset
17,6132,ONEBACK Correct Response
23,6221,TWOBACK Normal Trial Onset
24,6222,TWOBACK Hit Trial Onset
25,6223,TWOBACK Conflict Trial Onset


Trigger semantics audit PASSED.


In [27]:
# =============================================================
# Trigger selection used by FIGNN preprocessing
# =============================================================

SELECTED_EPOCH_TRIGGERS = {
    0: "6021",
    1: "6121",
    2: "6221"
}

print(
    "Selected event-aligned N-back onset triggers:"
)

for label, code in SELECTED_EPOCH_TRIGGERS.items():

    description = trigger_df.loc[
        trigger_df["code"] == code,
        "description"
    ].iloc[0]

    print(
        f"  label {label}: {code} -> {description}"
    )

print(
    "\nThese are normal-trial onset events; "
    "hit/conflict/response triggers are not used "
    "as epoch-start events."
)

Selected event-aligned N-back onset triggers:
  label 0: 6021 -> ZEROBACK Normal Trial Onset
  label 1: 6121 -> ONEBACK Normal Trial Onset
  label 2: 6221 -> TWOBACK Normal Trial Onset

These are normal-trial onset events; hit/conflict/response triggers are not used as epoch-start events.


## 18. Graph-Count and Trigger-Consistency Audit

This section verifies that the processed graph dataset has the expected
subject-, session-, and workload-level counts implied by the event-aligned
N-back preprocessing and the retained REST graphs.

In [33]:
# 18. Graph-Count and Provenance Audit

def scalar_attr(graph, name):
    value = getattr(graph, name, None)
    if value is None:
        raise AttributeError(f"Graph is missing required attribute: {name}")
    return int(torch.as_tensor(value).reshape(-1)[0].item())


# Actual attribute names used when graphs were built:
# graph.subject
# graph.session
# graph.y

graph_subjects = np.array([
    scalar_attr(g, "subject")
    for g in all_graphs
])

graph_sessions = np.array([
    scalar_attr(g, "session")
    for g in all_graphs
])

graph_labels = np.array([
    scalar_attr(g, "y")
    for g in all_graphs
])


# ------------------------------------------------------------
# Basic totals
# ------------------------------------------------------------

assert len(all_graphs) == 27666, (
    f"Expected 27666 graphs, found {len(all_graphs)}"
)

label_counts = {
    int(label): int(np.sum(graph_labels == label))
    for label in sorted(np.unique(graph_labels))
}

print("Total graphs:", len(all_graphs))
print("Label counts:", label_counts)


# Expected:
# 0-Back = 8352
# 1-Back = 8352
# 2-Back = 8352
# REST   = 2610

assert label_counts == {
    0: 8352,
    1: 8352,
    2: 8352,
    3: 2610,
}, f"Unexpected label counts: {label_counts}"


# ------------------------------------------------------------
# Per-subject audit
# ------------------------------------------------------------

for subject in range(1, 30):
    mask = graph_subjects == subject

    counts = {
        label: int(np.sum(mask & (graph_labels == label)))
        for label in [0, 1, 2, 3]
    }

    assert counts == {
        0: 288,
        1: 288,
        2: 288,
        3: 90,
    }, f"Subject {subject}: unexpected counts {counts}"


# ------------------------------------------------------------
# Per subject/session audit
# ------------------------------------------------------------

for subject in range(1, 30):
    for session in [1, 2, 3]:
        mask = (
            (graph_subjects == subject) &
            (graph_sessions == session)
        )

        counts = {
            label: int(np.sum(mask & (graph_labels == label)))
            for label in [0, 1, 2, 3]
        }

        assert counts == {
            0: 96,
            1: 96,
            2: 96,
            3: 30,
        }, (
            f"Subject {subject}, session {session}: "
            f"unexpected counts {counts}"
        )


print("PASS: 27,666 graphs")
print("PASS: 29 subjects × 954 graphs")
print("PASS: each subject = 288/288/288/90")
print("PASS: each subject/session = 96/96/96/30")

Total graphs: 27666
Label counts: {0: 8352, 1: 8352, 2: 8352, 3: 2610}
PASS: 27,666 graphs
PASS: 29 subjects × 954 graphs
PASS: each subject = 288/288/288/90
PASS: each subject/session = 96/96/96/30


## 19. COG-BCI Behavioral Audit

The COG-BCI N-back files are MATLAB MCOS `table` objects. This section uses
`mat-io` to decode those tables directly instead of manually probing the
serialized MCOS byte stream.


In [5]:
from pathlib import Path
import importlib.util
import pandas as pd

assert importlib.util.find_spec("matio") is not None, (
    "Missing package 'mat-io'. Install once with: pip install mat-io"
)

from matio import load_from_mat

BEHAVIORAL_ROOT = RAW
print("Behavioral root:", BEHAVIORAL_ROOT)
print("mat-io: available")


Behavioral root: D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw
mat-io: available


In [30]:
# 19A. Single-file decoder sanity check

def load_nback_table(mat_path):
    data = load_from_mat(str(mat_path), raw_data=False)
    frames = [v for v in data.values() if isinstance(v, pd.DataFrame)]

    if len(frames) != 1:
        raise RuntimeError(
            f"Expected exactly one decoded table in {mat_path}, found {len(frames)}"
        )

    df = frames[0].copy()
    df.columns = [str(c) for c in df.columns]
    return df

test_path = (
    BEHAVIORAL_ROOT
    / "sub-12"
    / "ses-S1"
    / "behavioral"
    / "0-Back.mat"
)

test_df = load_nback_table(test_path)

print("File:", test_path)
print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())
display(test_df.head())


File: D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\sub-12\ses-S1\behavioral\0-Back.mat
Shape: (144, 13)
Columns: ['block', 'rt', 'bin', 'hittrials', 'sbj', 'sess', 'miss', 'error', 'mistake', 'correct', 'condition', 'class', 'outlier']


,block,rt,bin,hittrials,sbj,sess,miss,error,mistake,correct,condition,class,outlier
0,1.0,NaN,0.0,0.0,12.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1.0,NaN,0.0,0.0,12.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,1.0,NaN,0.0,0.0,12.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,305.0,1.0,1.0,12.0,1.0,0.0,0.0,0.0,1.0,0.0,22.0,0.0
4,1.0,NaN,0.0,0.0,12.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [31]:
# 19B. Behavioral inventory for locally available subjects

available_subjects = [1, 12]
behavioral_rows = []

for subject in available_subjects:
    for session in ["S1", "S2", "S3"]:
        for condition in [0, 1, 2]:
            task = f"{condition}-Back"
            p = (
                BEHAVIORAL_ROOT
                / f"sub-{subject:02d}"
                / f"ses-{session}"
                / "behavioral"
                / f"{task}.mat"
            )

            assert p.exists(), f"Missing: {p}"

            df = load_nback_table(p)

            behavioral_rows.append({
                "subject": subject,
                "session": session,
                "condition": condition,
                "file": str(p),
                "n_rows": len(df),
                "columns": ", ".join(df.columns),
            })

behavioral_inventory = pd.DataFrame(behavioral_rows)
display(behavioral_inventory)

print("PASS: all locally available S01/S12 N-back files decode as tables.")


,subject,session,condition,file,n_rows,columns
0,1,S1,0,D:\Portfolio Projects\QML EEG Paper\fignn_cni_...,144,"block, rt, bin, hittrials, sbj, sess, miss, er..."
1,1,S1,1,D:\Portfolio Projects\QML EEG Paper\fignn_cni_...,144,"block, rt, bin, hittrials, sbj, sess, miss, er..."
2,1,S1,2,D:\Portfolio Projects\QML EEG Paper\fignn_cni_...,144,"block, rt, bin, hittrials, sbj, sess, miss, er..."
3,1,S2,0,D:\Portfolio Projects\QML EEG Paper\fignn_cni_...,144,"block, rt, bin, hittrials, sbj, sess, miss, er..."
4,1,S2,1,D:\Portfolio Projects\QML EEG Paper\fignn_cni_...,144,"block, rt, bin, hittrials, sbj, sess, miss, er..."
5,1,S2,2,D:\Portfolio Projects\QML EEG Paper\fignn_cni_...,144,"block, rt, bin, hittrials, sbj, sess, miss, er..."
6,1,S3,0,D:\Portfolio Projects\QML EEG Paper\fignn_cni_...,144,"block, rt, bin, hittrials, sbj, sess, miss, er..."
7,1,S3,1,D:\Portfolio Projects\QML EEG Paper\fignn_cni_...,144,"block, rt, bin, hittrials, sbj, sess, miss, er..."
8,1,S3,2,D:\Portfolio Projects\QML EEG Paper\fignn_cni_...,144,"block, rt, bin, hittrials, sbj, sess, miss, er..."
9,12,S1,0,D:\Portfolio Projects\QML EEG Paper\fignn_cni_...,144,"block, rt, bin, hittrials, sbj, sess, miss, er..."


PASS: all locally available S01/S12 N-back files decode as tables.


In [34]:
# 19C. Behavioral schema audit

expected_behavioral_columns = {
    "block", "rt", "bin", "hittrials", "sbj", "sess",
    "miss", "error", "mistake", "correct", "condition",
    "class", "outlier"
}

decoded_columns = set(test_df.columns)
missing_columns = expected_behavioral_columns - decoded_columns

print("Decoded columns:")
print(test_df.columns.tolist())

assert not missing_columns, (
    f"Missing expected behavioral columns: {missing_columns}"
)

print("PASS: expected COG-BCI N-back behavioral schema is present.")


Decoded columns:
['block', 'rt', 'bin', 'hittrials', 'sbj', 'sess', 'miss', 'error', 'mistake', 'correct', 'condition', 'class', 'outlier']
PASS: expected COG-BCI N-back behavioral schema is present.


In [35]:
# 19D. Behavioral value audit

behavioral_summary = []

for _, row in behavioral_inventory.iterrows():
    df = load_nback_table(Path(row["file"]))

    # Numeric conversion
    rt = pd.to_numeric(df["rt"], errors="coerce")

    behavioral_summary.append({
        "subject": row["subject"],
        "session": row["session"],
        "condition": row["condition"],
        "n_rows": len(df),
        "rt_valid": int(rt.notna().sum()),
        "rt_mean": float(rt.mean()) if rt.notna().any() else np.nan,
        "correct_sum": float(pd.to_numeric(df["correct"], errors="coerce").sum()),
        "error_sum": float(pd.to_numeric(df["error"], errors="coerce").sum()),
        "miss_sum": float(pd.to_numeric(df["miss"], errors="coerce").sum()),
        "mistake_sum": float(pd.to_numeric(df["mistake"], errors="coerce").sum()),
        "outlier_sum": float(pd.to_numeric(df["outlier"], errors="coerce").sum()),
    })

behavioral_summary = pd.DataFrame(behavioral_summary)

display(behavioral_summary)

print("PASS: behavioral values successfully extracted.")

,subject,session,condition,n_rows,rt_valid,rt_mean,correct_sum,error_sum,miss_sum,mistake_sum,outlier_sum
0,1,S1,0,144,48,394.500000,47.0,1.0,1.0,2.0,3.0
1,1,S1,1,144,46,461.739130,46.0,0.0,2.0,2.0,3.0
2,1,S1,2,144,47,607.638298,46.0,2.0,2.0,4.0,0.0
3,1,S2,0,144,50,424.460000,48.0,2.0,0.0,2.0,2.0
4,1,S2,1,144,47,462.489362,47.0,3.0,2.0,5.0,5.0
5,1,S2,2,144,51,681.274510,48.0,3.0,0.0,3.0,2.0
6,1,S3,0,144,48,393.541667,47.0,1.0,1.0,2.0,5.0
7,1,S3,1,144,47,391.659574,46.0,1.0,2.0,3.0,0.0
8,1,S3,2,144,49,557.938776,46.0,3.0,2.0,5.0,0.0
9,12,S1,0,144,31,593.193548,26.0,5.0,22.0,27.0,9.0


PASS: behavioral values successfully extracted.


In [40]:
# ============================================================
# 19E. FULL-COHORT COG-BCI BEHAVIORAL EXTRACTION
# ============================================================

import struct
import zlib
import time
import requests
from matio import load_from_mat

ZENODO_BASE = "https://zenodo.org/records/7413650/files"

BEHAVIORAL_DIR = RAW / "behavioral_full_cohort"
BEHAVIORAL_DIR.mkdir(parents=True, exist_ok=True)

BEHAVIORAL_CONDITIONS = {
    "0-Back": "0-Back.mat",
    "1-Back": "1-Back.mat",
    "2-Back": "2-Back.mat",
}

FAILED_SUBJECTS = {12, 14, 19, 22, 25, 29}

EXPECTED_BEHAVIORAL_COLUMNS = {
    "block", "rt", "bin", "hittrials", "sbj", "sess",
    "miss", "error", "mistake", "correct",
    "condition", "class", "outlier"
}

print("Output:", BEHAVIORAL_DIR)
print("Subjects:", len(SUBJECTS))
print("Expected MAT files:", len(SUBJECTS) * 3 * 3)

Output: D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\data\raw\behavioral_full_cohort
Subjects: 29
Expected MAT files: 261


In [42]:
# ============================================================
# 19F. REMOTE ZIP RANGE EXTRACTION
# ============================================================

http = requests.Session()


def range_request(url, start=None, end=None, retries=4):

    headers = {}

    if start is not None and end is not None:
        headers["Range"] = f"bytes={start}-{end}"
    elif start is not None:
        headers["Range"] = f"bytes={start}-"
    elif end is not None:
        headers["Range"] = f"bytes=-{end}"

    for attempt in range(retries):
        try:
            r = http.get(url, headers=headers, timeout=60)
            r.raise_for_status()
            return r.content

        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2)


def get_remote_size(url):

    r = http.head(url, timeout=60)
    r.raise_for_status()
    return int(r.headers["Content-Length"])


def parse_zip_central_directory(url):

    total_size = get_remote_size(url)

    # EOCD is normally inside the final 64 KB.
    tail_size = min(total_size, 1024 * 1024)

    tail = range_request(
        url,
        start=total_size - tail_size,
        end=total_size - 1
    )

    eocd_pos = tail.rfind(b"PK\x05\x06")

    if eocd_pos < 0:
        raise RuntimeError("ZIP EOCD not found.")

    (
        signature,
        disk_no,
        cd_disk,
        entries_disk,
        total_entries,
        cd_size,
        cd_offset,
        comment_len,
    ) = struct.unpack_from(
        "<4s4H2LH",
        tail,
        eocd_pos
    )

    if total_entries == 0xFFFF or cd_offset == 0xFFFFFFFF:
        raise RuntimeError("ZIP64 archive detected.")

    cd = range_request(
        url,
        start=cd_offset,
        end=cd_offset + cd_size - 1
    )

    entries = {}
    pos = 0

    while pos < len(cd):

        if cd[pos:pos + 4] != b"PK\x01\x02":
            break

        (
            sig,
            version_made,
            version_needed,
            flags,
            compression,
            mod_time,
            mod_date,
            crc32,
            compressed_size,
            uncompressed_size,
            filename_len,
            extra_len,
            comment_len,
            disk_start,
            internal_attr,
            external_attr,
            local_header_offset,
        ) = struct.unpack_from(
            "<4s6H3L5H2L",
            cd,
            pos
        )

        name_start = pos + 46
        name_end = name_start + filename_len

        filename = cd[name_start:name_end].decode("utf-8")

        entries[filename] = {
            "compression": compression,
            "compressed_size": compressed_size,
            "uncompressed_size": uncompressed_size,
            "local_header_offset": local_header_offset,
            "flags": flags,
        }

        pos = (
            name_end
            + extra_len
            + comment_len
        )

    return total_size, entries

In [47]:
import struct
import zlib
import time
import requests

from matio import load_from_mat


ZENODO_FILE_URL = "https://zenodo.org/records/7413650/files/{subject}.zip?download=1"

BEHAVIORAL_DIR.mkdir(parents=True, exist_ok=True)


def normalize_subject(subject):
    if isinstance(subject, int):
        return f"sub-{subject:02d}"

    subject = str(subject)

    if subject.startswith("sub-"):
        return subject

    return f"sub-{int(subject):02d}"


def get_zip_central_directory(url):
    response = requests.get(
        url,
        headers={"Range": "bytes=-1048576"},
        timeout=60
    )

    response.raise_for_status()

    tail = response.content

    eocd_signature = b"PK\x05\x06"
    eocd_offset = tail.rfind(eocd_signature)

    if eocd_offset == -1:
        raise RuntimeError("Could not find ZIP end-of-central-directory record")

    total_entries = struct.unpack_from("<H", tail, eocd_offset + 10)[0]
    central_size = struct.unpack_from("<I", tail, eocd_offset + 12)[0]
    central_offset = struct.unpack_from("<I", tail, eocd_offset + 16)[0]

    return total_entries, central_size, central_offset


def get_zip_entries(url):
    total_entries, central_size, central_offset = get_zip_central_directory(url)

    response = requests.get(
        url,
        headers={
            "Range": f"bytes={central_offset}-{central_offset + central_size - 1}"
        },
        timeout=120
    )

    response.raise_for_status()

    central_data = response.content
    entries = {}

    offset = 0

    for _ in range(total_entries):
        if central_data[offset:offset + 4] != b"PK\x01\x02":
            raise RuntimeError("Invalid ZIP central-directory entry")

        compressed_size = struct.unpack_from("<I", central_data, offset + 20)[0]
        uncompressed_size = struct.unpack_from("<I", central_data, offset + 24)[0]
        filename_length = struct.unpack_from("<H", central_data, offset + 28)[0]
        extra_length = struct.unpack_from("<H", central_data, offset + 30)[0]
        comment_length = struct.unpack_from("<H", central_data, offset + 32)[0]
        compression_method = struct.unpack_from("<H", central_data, offset + 10)[0]
        local_header_offset = struct.unpack_from("<I", central_data, offset + 42)[0]

        filename_start = offset + 46
        filename_end = filename_start + filename_length

        filename = central_data[filename_start:filename_end].decode(
            "utf-8",
            errors="replace"
        )

        entries[filename] = {
            "compressed_size": compressed_size,
            "uncompressed_size": uncompressed_size,
            "compression_method": compression_method,
            "local_header_offset": local_header_offset
        }

        offset = filename_end + extra_length + comment_length

    return entries


def extract_zip_member(url, info):
    local_header_offset = info["local_header_offset"]

    response = requests.get(
        url,
        headers={
            "Range": f"bytes={local_header_offset}-{local_header_offset + 4095}"
        },
        timeout=60
    )

    response.raise_for_status()

    header = response.content

    if header[:4] != b"PK\x03\x04":
        raise RuntimeError("Invalid ZIP local-file header")

    filename_length = struct.unpack_from("<H", header, 26)[0]
    extra_length = struct.unpack_from("<H", header, 28)[0]

    data_start = local_header_offset + 30 + filename_length + extra_length
    data_end = data_start + info["compressed_size"] - 1

    response = requests.get(
        url,
        headers={"Range": f"bytes={data_start}-{data_end}"},
        timeout=120
    )

    response.raise_for_status()

    compressed_data = response.content

    if info["compression_method"] == 0:
        return compressed_data

    if info["compression_method"] == 8:
        return zlib.decompress(compressed_data, -15)

    raise RuntimeError(
        f"Unsupported ZIP compression method: {info['compression_method']}"
    )


def extract_behavioral_mat(subject, session_name, mat_name):
    subject_name = normalize_subject(subject)

    output_dir = BEHAVIORAL_DIR / subject_name / session_name
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / mat_name

    if output_path.exists():
        return output_path

    url = ZENODO_FILE_URL.format(subject=subject_name)

    print(f"Reading ZIP structure: {subject_name}")

    zip_entries = get_zip_entries(url)

    target_suffix = f"ses-{session_name}/behavioral/{mat_name}"

    matching_members = [
        name
        for name in zip_entries
        if name.endswith(target_suffix)
        and name.lower().endswith(".mat")
    ]

    if len(matching_members) == 0:
        raise FileNotFoundError(
            f"No ZIP member ending with '{target_suffix}' found"
        )

    if len(matching_members) > 1:
        raise RuntimeError(
            f"Multiple matches for {target_suffix}: {matching_members}"
        )

    member = matching_members[0]
    info = zip_entries[member]

    print(f"  Extracting: {member}")

    mat_bytes = extract_zip_member(url, info)

    output_path.write_bytes(mat_bytes)

    return output_path


behavioral_records = []

for subject in SUBJECTS:
    subject_name = normalize_subject(subject)

    print(f"\n{'=' * 70}")
    print(f"SUBJECT: {subject_name}")
    print(f"{'=' * 70}")

    for session_name in ["S1", "S2", "S3"]:
        for mat_name in ["0-Back.mat", "1-Back.mat", "2-Back.mat"]:
            output_path = extract_behavioral_mat(
                subject,
                session_name,
                mat_name
            )

            behavioral_records.append({
                "subject": subject_name,
                "session": session_name,
                "condition": mat_name.replace("-Back.mat", ""),
                "path": str(output_path)
            })

            print(f"  OK: {session_name}/{mat_name}")

            time.sleep(0.1)


behavioral_manifest = pd.DataFrame(behavioral_records)

print("\nExtraction complete.")
print(f"Total MAT files: {len(behavioral_manifest)}")
print(f"Subjects: {behavioral_manifest['subject'].nunique()}")
print(f"Sessions: {behavioral_manifest['session'].nunique()}")
print(f"Conditions: {behavioral_manifest['condition'].nunique()}")


SUBJECT: sub-01
  OK: S1/0-Back.mat
  OK: S1/1-Back.mat
  OK: S1/2-Back.mat
  OK: S2/0-Back.mat
  OK: S2/1-Back.mat
  OK: S2/2-Back.mat
  OK: S3/0-Back.mat
  OK: S3/1-Back.mat
  OK: S3/2-Back.mat

SUBJECT: sub-02
  OK: S1/0-Back.mat
  OK: S1/1-Back.mat
  OK: S1/2-Back.mat
  OK: S2/0-Back.mat
  OK: S2/1-Back.mat
  OK: S2/2-Back.mat
  OK: S3/0-Back.mat
  OK: S3/1-Back.mat
  OK: S3/2-Back.mat

SUBJECT: sub-03
  OK: S1/0-Back.mat
Reading ZIP structure: sub-03
  Extracting: sub-03/sub-03/ses-S1/behavioral/1-Back.mat
  OK: S1/1-Back.mat
Reading ZIP structure: sub-03
  Extracting: sub-03/sub-03/ses-S1/behavioral/2-Back.mat
  OK: S1/2-Back.mat
Reading ZIP structure: sub-03
  Extracting: sub-03/sub-03/ses-S2/behavioral/0-Back.mat
  OK: S2/0-Back.mat
Reading ZIP structure: sub-03
  Extracting: sub-03/sub-03/ses-S2/behavioral/1-Back.mat
  OK: S2/1-Back.mat
Reading ZIP structure: sub-03
  Extracting: sub-03/sub-03/ses-S2/behavioral/2-Back.mat
  OK: S2/2-Back.mat
Reading ZIP structure: sub-03
  Ex

In [48]:
# ============================================================
# 19G. DECODE FULL-COHORT BEHAVIORAL TABLES
# ============================================================

behavioral_tables = []

for _, row in behavioral_manifest.iterrows():
    df = load_nback_table(Path(row["path"]))

    df["subject"] = int(row["subject"].replace("sub-", ""))
    df["session"] = row["session"]
    df["condition"] = int(row["condition"])

    behavioral_tables.append(df)

behavioral_all = pd.concat(
    behavioral_tables,
    ignore_index=True
)

print("Total behavioral rows:", len(behavioral_all))
print("Subjects:", behavioral_all["subject"].nunique())
print("Sessions:", behavioral_all["session"].nunique())
print("Conditions:", behavioral_all["condition"].nunique())

display(
    behavioral_all[
        [
            "subject",
            "session",
            "condition",
            "rt",
            "correct",
            "error",
            "miss",
            "mistake",
            "outlier"
        ]
    ].head()
)

Total behavioral rows: 37584
Subjects: 29
Sessions: 3
Conditions: 3


,subject,session,condition,rt,correct,error,miss,mistake,outlier
0,1,S1,0,NaN,0.0,0.0,0.0,0.0,0.0
1,1,S1,0,NaN,0.0,0.0,0.0,0.0,0.0
2,1,S1,0,NaN,0.0,0.0,0.0,0.0,0.0
3,1,S1,0,NaN,0.0,0.0,0.0,0.0,0.0
4,1,S1,0,277.0,1.0,0.0,0.0,0.0,0.0


In [49]:
# ============================================================
# 19H. SUBJECT-LEVEL BEHAVIORAL METRICS
# ============================================================

numeric_behavior_cols = [
    "rt",
    "correct",
    "error",
    "miss",
    "mistake",
    "outlier"
]

for col in numeric_behavior_cols:
    behavioral_all[col] = pd.to_numeric(
        behavioral_all[col],
        errors="coerce"
    )

behavioral_session = (
    behavioral_all
    .groupby(
        ["subject", "session", "condition"],
        as_index=False
    )
    .agg(
        n_trials=("subject", "size"),
        rt_mean=("rt", "mean"),
        rt_median=("rt", "median"),
        correct_mean=("correct", "mean"),
        error_mean=("error", "mean"),
        miss_mean=("miss", "mean"),
        mistake_mean=("mistake", "mean"),
        outlier_mean=("outlier", "mean"),
        correct_sum=("correct", "sum"),
        error_sum=("error", "sum"),
        miss_sum=("miss", "sum"),
        mistake_sum=("mistake", "sum"),
        outlier_sum=("outlier", "sum")
    )
)

print("Rows:", len(behavioral_session))
print("Expected rows:", 29 * 3 * 3)

assert len(behavioral_session) == 261

display(behavioral_session.head(12))

Rows: 261
Expected rows: 261


,subject,session,condition,n_trials,rt_mean,rt_median,correct_mean,error_mean,miss_mean,mistake_mean,outlier_mean,correct_sum,error_sum,miss_sum,mistake_sum,outlier_sum
0,1,S1,0,144,394.500000,367.5,0.326389,0.006944,0.006944,0.013889,0.020833,47.0,1.0,1.0,2.0,3.0
1,1,S1,1,144,461.739130,416.5,0.319444,0.000000,0.013889,0.013889,0.020833,46.0,0.0,2.0,2.0,3.0
2,1,S1,2,144,607.638298,579.0,0.319444,0.013889,0.013889,0.027778,0.000000,46.0,2.0,2.0,4.0,0.0
3,1,S2,0,144,424.460000,400.5,0.333333,0.013889,0.000000,0.013889,0.013889,48.0,2.0,0.0,2.0,2.0
4,1,S2,1,144,462.489362,438.0,0.326389,0.020833,0.013889,0.034722,0.034722,47.0,3.0,2.0,5.0,5.0
5,1,S2,2,144,681.274510,668.0,0.333333,0.020833,0.000000,0.020833,0.013889,48.0,3.0,0.0,3.0,2.0
6,1,S3,0,144,393.541667,354.0,0.326389,0.006944,0.006944,0.013889,0.034722,47.0,1.0,1.0,2.0,5.0
7,1,S3,1,144,391.659574,367.0,0.319444,0.006944,0.013889,0.020833,0.000000,46.0,1.0,2.0,3.0,0.0
8,1,S3,2,144,557.938776,550.0,0.319444,0.020833,0.013889,0.034722,0.000000,46.0,3.0,2.0,5.0,0.0
9,2,S1,0,144,366.250000,361.0,0.333333,0.006944,0.006944,0.013889,0.000000,48.0,1.0,1.0,2.0,0.0


In [50]:
# ============================================================
# 19I. SUBJECT-LEVEL WORKLOAD BEHAVIORAL PROFILE
# ============================================================

behavioral_subject = (
    behavioral_session
    .groupby(
        ["subject", "condition"],
        as_index=False
    )
    .agg(
        n_trials=("n_trials", "sum"),
        rt_mean=("rt_mean", "mean"),
        rt_median=("rt_median", "mean"),
        correct_mean=("correct_mean", "mean"),
        error_mean=("error_mean", "mean"),
        miss_mean=("miss_mean", "mean"),
        mistake_mean=("mistake_mean", "mean"),
        outlier_mean=("outlier_mean", "mean")
    )
)

behavioral_subject["group"] = np.where(
    behavioral_subject["subject"].isin(FAILED_SUBJECTS),
    "Failure",
    "Pass"
)

print("Subject-condition rows:", len(behavioral_subject))
print(
    behavioral_subject.groupby("group")["subject"]
    .nunique()
)

display(
    behavioral_subject
    .sort_values(["subject", "condition"])
)

Subject-condition rows: 87
group
Failure     6
Pass       23
Name: subject, dtype: int64


,subject,condition,n_trials,rt_mean,rt_median,correct_mean,error_mean,miss_mean,mistake_mean,outlier_mean,group
0,1,0,432,404.167222,374.000000,0.328704,0.009259,0.004630,0.013889,0.023148,Pass
1,1,1,432,438.629356,407.166667,0.321759,0.009259,0.013889,0.023148,0.018519,Pass
2,1,2,432,615.617194,599.000000,0.324074,0.018519,0.009259,0.027778,0.004630,Pass
3,2,0,432,360.076531,355.333333,0.333333,0.006944,0.002315,0.009259,0.002315,Pass
4,2,1,432,357.088939,347.166667,0.331019,0.011574,0.002315,0.013889,0.000000,Pass
...,...,...,...,...,...,...,...,...,...,...,...
82,28,1,432,406.778382,389.166667,0.324074,0.004630,0.009259,0.013889,0.002315,Pass
83,28,2,432,557.769213,526.333333,0.296296,0.039352,0.037037,0.076389,0.009259,Pass
84,29,0,432,371.760062,363.333333,0.333333,0.004630,0.000000,0.004630,0.009259,Failure
85,29,1,432,388.465278,373.166667,0.328704,0.000000,0.004630,0.004630,0.000000,Failure


In [51]:
# ============================================================
# 19J. WIDE SUBJECT-LEVEL BEHAVIORAL PROFILE
# ============================================================

behavioral_wide = behavioral_subject.pivot(
    index=["subject", "group"],
    columns="condition",
    values=[
        "rt_mean",
        "rt_median",
        "correct_mean",
        "error_mean",
        "miss_mean",
        "mistake_mean",
        "outlier_mean"
    ]
)

behavioral_wide.columns = [
    f"{metric}_{condition}back"
    for metric, condition in behavioral_wide.columns
]

behavioral_wide = behavioral_wide.reset_index()

behavioral_wide = behavioral_wide.sort_values("subject")

print("Subjects:", len(behavioral_wide))

display(behavioral_wide)

Subjects: 29


,subject,group,rt_mean_0back,rt_mean_1back,rt_mean_2back,rt_median_0back,rt_median_1back,rt_median_2back,correct_mean_0back,correct_mean_1back,...,error_mean_2back,miss_mean_0back,miss_mean_1back,miss_mean_2back,mistake_mean_0back,mistake_mean_1back,mistake_mean_2back,outlier_mean_0back,outlier_mean_1back,outlier_mean_2back
0,1,Pass,404.167222,438.629356,615.617194,374.000000,407.166667,599.000000,0.328704,0.321759,...,0.018519,0.004630,0.013889,0.009259,0.013889,0.023148,0.027778,0.023148,0.018519,0.004630
1,2,Pass,360.076531,357.088939,491.241297,355.333333,347.166667,460.666667,0.333333,0.331019,...,0.023148,0.002315,0.002315,0.011574,0.009259,0.013889,0.034722,0.002315,0.000000,0.002315
2,3,Pass,375.378827,391.846982,570.786667,368.166667,373.833333,550.666667,0.333333,0.333333,...,0.016204,0.000000,0.002315,0.002315,0.004630,0.006944,0.018519,0.002315,0.002315,0.006944
3,4,Pass,408.375000,489.589470,583.743114,398.000000,438.000000,546.666667,0.333333,0.324074,...,0.020833,0.000000,0.016204,0.018519,0.004630,0.023148,0.039352,0.004630,0.025463,0.009259
4,5,Pass,555.604167,709.180556,831.609162,556.666667,691.500000,784.833333,0.333333,0.333333,...,0.011574,0.000000,0.000000,0.106481,0.000000,0.000000,0.118056,0.150463,0.164352,0.030093
5,6,Pass,440.595238,552.830676,645.560652,406.666667,503.333333,513.500000,0.328704,0.307870,...,0.032407,0.004630,0.030093,0.108796,0.018519,0.046296,0.141204,0.037037,0.055556,0.023148
6,7,Pass,354.701956,431.662698,716.284586,346.833333,394.000000,647.666667,0.333333,0.333333,...,0.027778,0.000000,0.000000,0.032407,0.002315,0.004630,0.057870,0.004630,0.023148,0.032407
7,8,Pass,477.962252,526.381844,609.413326,447.333333,500.833333,576.500000,0.328704,0.321759,...,0.027778,0.004630,0.011574,0.050926,0.013889,0.032407,0.078704,0.062500,0.030093,0.006944
8,9,Pass,403.457625,454.574320,676.387288,386.000000,429.333333,674.666667,0.328704,0.326389,...,0.034722,0.004630,0.006944,0.039352,0.011574,0.011574,0.074074,0.016204,0.006944,0.006944
9,10,Pass,379.799722,387.538124,450.066468,362.333333,372.166667,405.500000,0.333333,0.333333,...,0.011574,0.000000,0.000000,0.009259,0.004630,0.002315,0.018519,0.004630,0.000000,0.000000


In [52]:
# ============================================================
# 19K. MERGE BEHAVIORAL AND FIGNN GEOMETRIC RESULTS
# ============================================================

geometry = stage1[
    [
        "subject",
        "relative_1back_position",
        "d01",
        "d02",
        "d12",
        "path_efficiency",
        "procrustes_similarity",
        "direction_cosine",
        "perpendicular_distance",
        "primary_pass"
    ]
].copy()

behavioral_geometry = behavioral_wide.merge(
    geometry,
    on="subject",
    how="left"
)

assert len(behavioral_geometry) == 29
assert behavioral_geometry["relative_1back_position"].notna().all()

display(
    behavioral_geometry[
        [
            "subject",
            "group",
            "relative_1back_position",
            "rt_mean_0back",
            "rt_mean_1back",
            "rt_mean_2back",
            "correct_mean_0back",
            "correct_mean_1back",
            "correct_mean_2back"
        ]
    ]
)

,subject,group,relative_1back_position,rt_mean_0back,rt_mean_1back,rt_mean_2back,correct_mean_0back,correct_mean_1back,correct_mean_2back
0,1,Pass,0.419143,404.167222,438.629356,615.617194,0.328704,0.321759,0.324074
1,2,Pass,0.154232,360.076531,357.088939,491.241297,0.333333,0.331019,0.321759
2,3,Pass,0.288697,375.378827,391.846982,570.786667,0.333333,0.333333,0.331019
3,4,Pass,0.620354,408.375000,489.589470,583.743114,0.333333,0.324074,0.314815
4,5,Pass,0.484313,555.604167,709.180556,831.609162,0.333333,0.333333,0.226852
5,6,Pass,0.469133,440.595238,552.830676,645.560652,0.328704,0.307870,0.224537
6,7,Pass,0.299064,354.701956,431.662698,716.284586,0.333333,0.333333,0.305556
7,8,Pass,0.427003,477.962252,526.381844,609.413326,0.328704,0.321759,0.282407
8,9,Pass,0.002998,403.457625,454.574320,676.387288,0.328704,0.326389,0.293981
9,10,Pass,0.120020,379.799722,387.538124,450.066468,0.333333,0.333333,0.328704


In [53]:
# ============================================================
# 19L. PASS VS FAILURE BEHAVIORAL COMPARISON
# ============================================================

from scipy.stats import mannwhitneyu

comparison_metrics = [
    "rt_mean_0back",
    "rt_mean_1back",
    "rt_mean_2back",
    "correct_mean_0back",
    "correct_mean_1back",
    "correct_mean_2back",
    "error_mean_0back",
    "error_mean_1back",
    "error_mean_2back",
    "miss_mean_0back",
    "miss_mean_1back",
    "miss_mean_2back",
    "mistake_mean_0back",
    "mistake_mean_1back",
    "mistake_mean_2back",
    "outlier_mean_0back",
    "outlier_mean_1back",
    "outlier_mean_2back"
]

comparison_rows = []

for metric in comparison_metrics:
    fail_values = behavioral_geometry.loc[
        behavioral_geometry["group"] == "Failure",
        metric
    ].dropna()

    pass_values = behavioral_geometry.loc[
        behavioral_geometry["group"] == "Pass",
        metric
    ].dropna()

    stat, p_value = mannwhitneyu(
        fail_values,
        pass_values,
        alternative="two-sided"
    )

    comparison_rows.append({
        "metric": metric,
        "failure_mean": fail_values.mean(),
        "pass_mean": pass_values.mean(),
        "failure_median": fail_values.median(),
        "pass_median": pass_values.median(),
        "U": stat,
        "p_raw": p_value
    })

behavioral_comparison = pd.DataFrame(comparison_rows)

behavioral_comparison = behavioral_comparison.sort_values(
    "p_raw"
)

display(behavioral_comparison)

,metric,failure_mean,pass_mean,failure_median,pass_median,U,p_raw
6,error_mean_0back,0.021219,0.005636,0.010417,0.004630,102.5,0.072206
12,mistake_mean_0back,0.030093,0.007548,0.018519,0.004630,102.5,0.072637
8,error_mean_2back,0.032793,0.026067,0.026620,0.018519,86.5,0.357442
15,outlier_mean_0back,0.025463,0.018619,0.021991,0.004630,85.5,0.383149
14,mistake_mean_2back,0.064429,0.056763,0.060185,0.039352,80.5,0.553361
13,mistake_mean_1back,0.026235,0.016506,0.015046,0.009259,79.0,0.607072
0,rt_mean_0back,419.403424,402.384507,416.357073,392.433469,76.0,0.733144
17,outlier_mean_2back,0.016204,0.011473,0.009259,0.006944,75.5,0.744938
16,outlier_mean_1back,0.034722,0.020229,0.017361,0.009259,74.0,0.807427
11,miss_mean_2back,0.031636,0.030898,0.024306,0.025463,64.5,0.829077


In [54]:
# ============================================================
# 19M. HOLM CORRECTION
# ============================================================

from statsmodels.stats.multitest import multipletests

behavioral_comparison["p_holm"] = multipletests(
    behavioral_comparison["p_raw"],
    method="holm"
)[1]

behavioral_comparison["significant_holm"] = (
    behavioral_comparison["p_holm"] < 0.05
)

display(
    behavioral_comparison.sort_values("p_holm")
)

,metric,failure_mean,pass_mean,failure_median,pass_median,U,p_raw,p_holm,significant_holm
6,error_mean_0back,0.021219,0.005636,0.010417,0.004630,102.5,0.072206,1.0,False
12,mistake_mean_0back,0.030093,0.007548,0.018519,0.004630,102.5,0.072637,1.0,False
8,error_mean_2back,0.032793,0.026067,0.026620,0.018519,86.5,0.357442,1.0,False
15,outlier_mean_0back,0.025463,0.018619,0.021991,0.004630,85.5,0.383149,1.0,False
14,mistake_mean_2back,0.064429,0.056763,0.060185,0.039352,80.5,0.553361,1.0,False
13,mistake_mean_1back,0.026235,0.016506,0.015046,0.009259,79.0,0.607072,1.0,False
0,rt_mean_0back,419.403424,402.384507,416.357073,392.433469,76.0,0.733144,1.0,False
17,outlier_mean_2back,0.016204,0.011473,0.009259,0.006944,75.5,0.744938,1.0,False
16,outlier_mean_1back,0.034722,0.020229,0.017361,0.009259,74.0,0.807427,1.0,False
11,miss_mean_2back,0.031636,0.030898,0.024306,0.025463,64.5,0.829077,1.0,False


In [55]:
# ============================================================
# 19N. BEHAVIORAL ASSOCIATION WITH GEOMETRIC RHO
# ============================================================

from scipy.stats import spearmanr

rho_metrics = [
    "rt_mean_0back",
    "rt_mean_1back",
    "rt_mean_2back",
    "correct_mean_0back",
    "correct_mean_1back",
    "correct_mean_2back",
    "error_mean_0back",
    "error_mean_1back",
    "error_mean_2back",
    "miss_mean_0back",
    "miss_mean_1back",
    "miss_mean_2back",
    "mistake_mean_0back",
    "mistake_mean_1back",
    "mistake_mean_2back",
    "outlier_mean_0back",
    "outlier_mean_1back",
    "outlier_mean_2back"
]

association_rows = []

for metric in rho_metrics:
    x = behavioral_geometry[metric]
    y = behavioral_geometry["relative_1back_position"]

    valid = x.notna() & y.notna()

    r, p = spearmanr(
        x[valid],
        y[valid]
    )

    association_rows.append({
        "metric": metric,
        "spearman_rho": r,
        "p_raw": p,
        "n": int(valid.sum())
    })

behavioral_rho_assoc = pd.DataFrame(association_rows)

behavioral_rho_assoc["p_holm"] = multipletests(
    behavioral_rho_assoc["p_raw"],
    method="holm"
)[1]

behavioral_rho_assoc = behavioral_rho_assoc.sort_values(
    "p_holm"
)

display(behavioral_rho_assoc)

,metric,spearman_rho,p_raw,n,p_holm
0,rt_mean_0back,0.051232,0.791836,29,1.0
1,rt_mean_1back,0.124631,0.519476,29,1.0
2,rt_mean_2back,-0.038916,0.841146,29,1.0
3,correct_mean_0back,-0.050075,0.796438,29,1.0
4,correct_mean_1back,-0.111475,0.564815,29,1.0
5,correct_mean_2back,-0.257962,0.176677,29,1.0
6,error_mean_0back,-0.325506,0.084874,29,1.0
7,error_mean_1back,-0.007231,0.970305,29,1.0
8,error_mean_2back,-0.269940,0.156724,29,1.0
9,miss_mean_0back,-0.018691,0.923333,29,1.0


In [56]:
# ============================================================
# 19O. SIX GEOMETRIC FAILURE SUBJECTS
# ============================================================

failure_behavioral = behavioral_geometry[
    behavioral_geometry["subject"].isin(FAILED_SUBJECTS)
].copy()

failure_behavioral = failure_behavioral.sort_values("subject")

display(
    failure_behavioral[
        [
            "subject",
            "group",
            "relative_1back_position",
            "rt_mean_0back",
            "rt_mean_1back",
            "rt_mean_2back",
            "correct_mean_0back",
            "correct_mean_1back",
            "correct_mean_2back",
            "error_mean_0back",
            "error_mean_1back",
            "error_mean_2back",
            "miss_mean_0back",
            "miss_mean_1back",
            "miss_mean_2back"
        ]
    ]
)

,subject,group,relative_1back_position,rt_mean_0back,rt_mean_1back,rt_mean_2back,correct_mean_0back,correct_mean_1back,correct_mean_2back,error_mean_0back,error_mean_1back,error_mean_2back,miss_mean_0back,miss_mean_1back,miss_mean_2back
11,12,Failure,-0.065809,494.414006,691.251634,776.061297,0.282407,0.282407,0.233796,0.013889,0.004630,0.032407,0.050926,0.050926,0.099537
13,14,Failure,-0.003911,463.598498,528.107143,602.804280,0.333333,0.333333,0.331019,0.071759,0.067130,0.064815,0.000000,0.000000,0.002315
18,19,Failure,-0.329658,364.180839,375.180556,529.743475,0.333333,0.333333,0.326389,0.002315,0.000000,0.013889,0.000000,0.000000,0.006944
21,22,Failure,1.149353,361.513056,371.289966,427.567311,0.331019,0.328704,0.300926,0.006944,0.006944,0.020833,0.002315,0.004630,0.032407
24,25,Failure,-0.281329,460.954084,496.510497,832.264314,0.333333,0.324074,0.310185,0.027778,0.009259,0.048611,0.000000,0.009259,0.023148
28,29,Failure,-0.031756,371.760062,388.465278,553.400236,0.333333,0.328704,0.307870,0.004630,0.000000,0.016204,0.000000,0.004630,0.025463


In [57]:
# ============================================================
# 19P. SAVE BEHAVIORAL AUDIT RESULTS
# ============================================================

behavioral_all.to_csv(
    METADATA_OUT / "behavioral_all_trials.csv",
    index=False
)

behavioral_session.to_csv(
    METADATA_OUT / "behavioral_session_summary.csv",
    index=False
)

behavioral_subject.to_csv(
    METADATA_OUT / "behavioral_subject_condition_summary.csv",
    index=False
)

behavioral_geometry.to_csv(
    METADATA_OUT / "behavioral_geometry_combined.csv",
    index=False
)

behavioral_comparison.to_csv(
    METADATA_OUT / "behavioral_failure_vs_pass_comparison.csv",
    index=False
)

behavioral_rho_assoc.to_csv(
    METADATA_OUT / "behavioral_rho_associations.csv",
    index=False
)

failure_behavioral.to_csv(
    METADATA_OUT / "six_failure_subjects_behavioral.csv",
    index=False
)

print("Behavioral audit exports saved to:")
print(METADATA_OUT)

Behavioral audit exports saved to:
D:\Portfolio Projects\QML EEG Paper\fignn_cni__project\data\outputs\final_non_gpu_validation\dataset_metadata


## 20. Final audit/export


In [ ]:
audit = {
    "n_subjects": len(stage1),
    "primary_pass": int(stage1.primary_pass.sum()),
    "distance_order_pass": int(stage1.distance_order_pass.sum()),
    "mean_d01_d02": float(stage1.d01_d02.mean()),
    "mean_path_efficiency": float(stage1.path_efficiency.mean()),
    "mean_procrustes": float(stage1.procrustes_similarity.mean()),
    "mean_perpendicular_distance": float(
        stage1.perpendicular_distance.mean()
    )
}

print(json.dumps(audit, indent=2))

assert audit["n_subjects"] == 29
assert audit["primary_pass"] == 23
assert audit["distance_order_pass"] == 27

(EXP / "final_audit.json").write_text(
    json.dumps(audit, indent=2)
)

print("All exports:", EXP)